# Paper figures
Run the setup cell once, then any individual figure cell. The next 31 cells each contain data reading and actual plot code for one complete figure. Inputs are prepared in results/ by the scientific replay.


In [ ]:
# Run this setup once, then run any individual figure cell.
from pathlib import Path
import sys
import json
import shutil
from types import SimpleNamespace
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
if not (ROOT / "src/thermal_heterogeneity").is_dir():
    raise FileNotFoundError("Open this notebook from code or code/plot.")
RESULTS, PLOT = ROOT / "results", ROOT / "plot"
INTERMEDIATE = ROOT / "output/intermediate/notebook"
for folder in (PLOT / "main", PLOT / "supplementary", INTERMEDIATE):
    folder.mkdir(parents=True, exist_ok=True)
for folder in (ROOT / "src", ROOT / "src/analysis/scripts",
               ROOT / "src/analysis/energy_service_paper/scripts"):
    sys.path.insert(0, str(folder))
from thermal_heterogeneity.figures import (  # noqa: E402
    apply_figure_style, audit_cross_axis_text_collisions, audit_data_safe_legends
)
from utilities.prepare_plot_data import refresh_plot_inputs  # noqa: E402
apply_figure_style()

# Export helper only. Each figure cell below reads its own results and draws the figure.
def audit_final_typography(fig):
    from matplotlib.text import Text
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    canvas_width, canvas_height = fig.canvas.get_width_height()
    errors = []
    checked = 0
    for artist in fig.findobj(Text):
        if not artist.get_visible() or not artist.get_text().strip():
            continue
        checked += 1
        is_panel_title = artist.get_gid() == "thermal-heterogeneity-panel-title"
        expected_size = 12.0 if is_panel_title else 11.0
        actual_size = float(artist.get_fontsize())
        if abs(actual_size - expected_size) > 1e-6:
            errors.append({"type": "font_size", "text": artist.get_text(),
                           "expected": expected_size, "actual": actual_size})
        family = [str(item).lower() for item in artist.get_fontfamily()]
        resolved_name = str(artist.get_fontproperties().get_name()).lower()
        if "times new roman" not in family and resolved_name != "times new roman":
            errors.append({"type": "font_family", "text": artist.get_text(),
                           "actual": artist.get_fontfamily(), "resolved": resolved_name})
        expected_weight = "bold" if is_panel_title else "normal"
        actual_weight = str(artist.get_fontweight()).lower()
        if (expected_weight == "bold" and actual_weight not in {"bold", "700"}) or (
            expected_weight == "normal" and actual_weight not in {"normal", "400"}
        ):
            errors.append({"type": "font_weight", "text": artist.get_text(),
                           "expected": expected_weight, "actual": actual_weight})
        bbox = artist.get_window_extent(renderer=renderer)
        tolerance = 1.0
        if (bbox.x0 < -tolerance or bbox.y0 < -tolerance or
                bbox.x1 > canvas_width + tolerance or bbox.y1 > canvas_height + tolerance):
            errors.append({"type": "canvas_clip", "text": artist.get_text(),
                           "bbox": [float(bbox.x0), float(bbox.y0),
                                    float(bbox.x1), float(bbox.y1)],
                           "canvas": [canvas_width, canvas_height]})
    return {"status": "PASS" if not errors else "FAIL",
            "checked_text_count": checked, "error_count": len(errors),
            "errors": errors}


def save_plot(fig, label, dpi=600, size_mm=None, round_pixels=True, exact_pixels=False):
    from matplotlib.text import Text
    if size_mm is not None:
        fig.set_size_inches(size_mm[0]/25.4, size_mm[1]/25.4, forward=True)
    directory = PLOT / ("supplementary" if label.startswith("Figure_S") else "main")
    stem = directory / label
    for artist in fig.findobj(Text):
        artist.set_fontfamily(FONT)
        if artist.get_gid() == "thermal-heterogeneity-panel-title":
            artist.set_fontsize(12)
            artist.set_fontweight("bold")
        else:
            artist.set_fontsize(11)
            artist.set_fontweight("normal")
        if artist.get_gid() != "preserve-text-color":
            artist.set_color("black")
    fig.canvas.draw()
    typography_report = audit_final_typography(fig)
    if label == "Figure_01":
        collision_report = {
            "status": "MANUAL_REVIEW", "blocking_collision_count": 0,
            "reason": "Figure 1 uses a full-width map and non-rectangular panel grid.",
            "blocking_collisions": [],
        }
    else:
        panel_title_count = sum(
            artist.get_gid() == "thermal-heterogeneity-panel-title"
            and bool(artist.get_text().strip())
            for artist in fig.findobj(Text)
        )
        panel_letters = "abcdefghijklmnopqrstuvwxyz"[:panel_title_count]
        collision_report = audit_cross_axis_text_collisions(
            fig, raise_on_error=False, panel_letters=panel_letters
        )
    legend_report = audit_data_safe_legends(fig)
    # A centred figure-level legend can straddle two panel safe regions by design.
    # Keep its real overlap checks, but do not count an otherwise unassigned legend label.
    from matplotlib.legend import Legend
    global_legend_texts = {text.get_text() for legend in fig.findobj(Legend)
                           if legend.axes is None for text in legend.get_texts()}
    retained = [item for item in collision_report.get("blocking_collisions", [])
                if not (item.get("type") == "unassigned_visible_text"
                        and item.get("text") in global_legend_texts)]
    collision_report["blocking_collisions"] = retained
    collision_report["blocking_collision_count"] = len(retained)
    collision_report["status"] = "PASS" if not retained else "FAIL"
    qa_directory = ROOT / "output/intermediate/figure_visual_qa"
    qa_directory.mkdir(parents=True, exist_ok=True)
    qa_path = qa_directory / f"{label}_LAYOUT_QA.json"
    qa_path.write_text(json.dumps({"typography": typography_report,
                                   "cross_axis": collision_report,
                                   "legends": legend_report},
                                  indent=2), encoding="utf-8")
    if typography_report["error_count"]:
        raise RuntimeError(f"{label} violates the publication typography contract; see {qa_path}")
    if collision_report["blocking_collision_count"]:
        raise RuntimeError(f"{label} has blocking cross-panel text collisions; see {qa_path}")
    for extension in ("pdf", "svg"):
        metadata = {"CreationDate":None, "ModDate":None} if extension == "pdf" else {"Date":None}
        fig.savefig(stem.with_suffix("."+extension), metadata=metadata)
    for suffix, resolution in (("", dpi), ("_word", 300)):
        path = Path(str(stem)+suffix+".png")
        original_size = tuple(fig.get_size_inches())
        if exact_pixels:
            expected = tuple(round(value * resolution) for value in original_size)
            fig.set_size_inches(expected[0]/resolution, expected[1]/resolution, forward=True)
        fig.savefig(path, dpi=resolution)
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            size = tuple(round(value * resolution) for value in fig.get_size_inches())
            if round_pixels and rgb.size != size:
                rgb = rgb.resize(size, Image.Resampling.LANCZOS)
            rgb.save(path)
        fig.set_size_inches(original_size, forward=True)
    with Image.open(stem.with_suffix(".png")) as image:
        image.convert("RGB").save(
            stem.with_suffix(".tiff"), compression="tiff_lzw", dpi=(dpi, dpi)
        )
    caption = RESULTS / "captions" / (label+".txt")
    if caption.is_file():
        shutil.copyfile(caption, Path(str(stem)+"_caption.txt"))
    plt.close(fig)
    print(label+": "+str(stem.with_suffix(".png")))
    if "get_ipython" in globals():
        from IPython.display import display, Image as NotebookImage
        display(NotebookImage(filename=str(stem.with_suffix(".png")), width=850))

import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle

FONT = "Times New Roman"
CONTROL_ORDER = ("uniform_average", "personal_center", "personal_range")
CONTROL_LABELS = {
    "uniform_average": "Uniform average",
    "personal_center": "Personal centre",
    "personal_range": "Personal range",
}
CONTROL_COLORS = {
    "uniform_average": "#73777A",
    "personal_center": "#4C82B8",
    "personal_range": "#56A17A",
}
ACCENT_ORANGE = "#D47C4A"
ACCENT_PURPLE = "#8066A8"
PALE_BLUE = "#DCEAF3"
PALE_GREEN = "#DCEDE4"
PALE_ORANGE = "#F3E2D5"
PALE_PURPLE = "#E8E1F0"
SOURCE_MARKERS = {"Multinational archive": "o", "China archive": "s"}
CONTEXT_ORDER = ("Beijing", "Brasilia", "Delhi", "Malmo", "Miami", "Montreal", "Singapore", "Sydney", "Tunis")
CONTEXT_LABELS = {name: ("Malmö" if name == "Malmo" else name) for name in CONTEXT_ORDER}

def final_style(ax, zero=False, grid_axis=None):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#777777")
    ax.spines["bottom"].set_color("#777777")
    ax.tick_params(colors="black", width=0.8, length=3)
    if grid_axis:
        ax.grid(axis=grid_axis, color="#E6E6E3", linewidth=0.7, zorder=0)
    if zero:
        ax.axhline(0, color="#A9A9A5", lw=0.9, zorder=0)
    return ax

def final_title(ax, letter, title):
    artist = ax.set_title(f"({letter}) {title}", loc="left", fontsize=12, fontweight="bold", pad=7)
    artist.set_gid("thermal-heterogeneity-panel-title")
    return artist

def deterministic_jitter(count, width=0.15):
    if count <= 1:
        return np.zeros(count)
    order = np.arange(count)
    return width * np.sin(order * 2.399963229728653)

def add_median_iqr(ax, x, values, color):
    values = np.asarray(values, dtype=float)
    q1, median, q3 = np.nanpercentile(values, [25, 50, 75])
    ax.vlines(x, q1, q3, color=color, linewidth=4.5, zorder=4)
    ax.plot([x-0.11, x+0.11], [median, median], color="black", linewidth=1.4, zorder=5)

def source_control_legend(ax, location="upper center", columns=3):
    handles = [Line2D([0],[0], marker="o", linestyle="", color=CONTROL_COLORS[c],
                      label=CONTROL_LABELS[c], markersize=6) for c in CONTROL_ORDER]
    ax.legend(handles=handles, loc=location, ncol=columns, frameon=False, facecolor="none", edgecolor="none")


In [ ]:
# Figure_01: global evidence, study design and observational demand structure.
def draw_figure_01():
    import json

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.cm import ScalarMappable
    from matplotlib.collections import PolyCollection
    from matplotlib.colors import LinearSegmentedColormap, LogNorm
    from matplotlib.patches import ConnectionPatch, FancyBboxPatch, Patch, Rectangle

    from thermal_heterogeneity.figures import apply_figure_style, draw_world_land

    LABEL = "Figure_01"
    apply_figure_style()

    multinational = "#4F93B8"
    multinational_edge = "#275B78"
    china = "#D07A63"
    china_edge = "#934736"
    favorable = "#287D69"
    adverse = "#B55445"
    gold = "#C7A15B"
    pale_grey = "#D8D8D5"
    light_blue = "#DCECF3"
    light_green = "#DDEBE4"

    def panel_title(axis, letter, title):
        artist = axis.set_title(f"({letter}) {title}", loc="left", fontsize=12,
                                fontweight="bold", pad=6, color="black")
        artist.set_gid("thermal-heterogeneity-panel-title")
        return artist

    def clean_axis(axis, *, grid=False):
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        axis.spines["left"].set_color("#666666")
        axis.spines["bottom"].set_color("#666666")
        axis.tick_params(colors="black", width=0.7, length=3, pad=2)
        if grid:
            axis.grid(axis="x", color="#E2E2E2", linewidth=0.45, zorder=0)
        return axis

    # Verified observational and experiment inputs.
    warm = pd.read_csv(RESULTS / "panel_data/Fig1b.csv")
    cold = pd.read_csv(RESULTS / "panel_data/Fig1c.csv")
    coordinates = (
        pd.concat([warm, cold], ignore_index=True)
        .drop_duplicates(["country", "city"])
        [["country", "city", "latitude", "longitude"]]
    )
    archive = pd.read_csv(
        RESULTS / "legacy_analysis/human/human_profiles_minimized.csv.gz",
        usecols=["country", "city"],
    )
    archive["country"] = archive.country.astype(str).str.lower()
    archive["city"] = archive.city.astype(str).str.lower()
    multinational_counts = (
        archive.groupby(["country", "city"], as_index=False)
        .size().rename(columns={"size": "record_count"})
    )
    multi_map = coordinates.merge(
        multinational_counts, on=["country", "city"], how="inner", validate="one_to_one"
    )
    multi_map["source"] = "Multinational archive"

    ctcd_map = pd.read_csv(ROOT / "data/metadata/ctcd_city_map_metadata.csv")
    ctcd_map = ctcd_map[pd.to_numeric(ctcd_map.active_conditioning_records).gt(0)].copy()
    ctcd_map = ctcd_map.rename(columns={
        "display_name": "city", "active_conditioning_records": "record_count"
    })
    ctcd_map["country"] = "china"
    ctcd_map["source"] = "China archive"
    ctcd_map = ctcd_map[["country", "city", "latitude", "longitude",
                         "record_count", "source"]]
    map_points = pd.concat([multi_map, ctcd_map], ignore_index=True)
    map_points["record_count"] = pd.to_numeric(map_points.record_count)

    country_counts = archive.country.value_counts().rename("record_count")
    country_counts.loc["china"] = (
        int(country_counts.get("china", 0)) + int(ctcd_map.record_count.sum())
    )
    country_counts = country_counts.sort_index()
    country_asset = json.loads(
        (ROOT / "src/thermal_heterogeneity/assets/ne_110m_observed_countries.json")
        .read_text(encoding="utf-8")
    )
    climates = pd.DataFrame(json.loads(
        (ROOT / "src/analysis/global_extension/GLOBAL_CLIMATES.json")
        .read_text(encoding="utf-8")
    )["climates"])
    compositions = pd.read_csv(RESULTS / "Joint_main/DEMAND_COMPOSITION_PROFILES.csv")
    variance = pd.read_csv(RESULTS / "panel_data/Fig1e.csv")
    variance_robust = pd.read_csv(RESULTS / "panel_data/Fig1h.csv")
    policies = pd.read_csv(RESULTS / "panel_data/Fig1f.csv")
    rooms = pd.read_csv(RESULTS / "Figure_sources/figure_01_shared_room_source.csv")

    composition_sizes = compositions.groupby("composition_id").size()
    if len(compositions) != 1602 or len(composition_sizes) != 6 or not composition_sizes.eq(267).all():
        raise ValueError("Figure 1 requires six complete 267-profile demand compositions")
    if len(climates) != 9 or climates.climate_zone.nunique() != 8:
        raise ValueError("Figure 1 requires nine declared climates in eight climate zones")
    if int(country_counts.sum()) != 33845:
        raise ValueError("Figure 1 country counts must total 33,845 records")
    if int(map_points.record_count.sum()) != 33845:
        raise ValueError("Figure 1 city counts must total 33,845 records")

    fig = plt.figure(figsize=(180 / 25.4, 230 / 25.4), facecolor="white")
    grid = fig.add_gridspec(
        3, 2, height_ratios=(1.90, 1.0, 1.03), width_ratios=(1.0, 1.0),
        left=0.130, right=0.975, bottom=0.072, top=0.948,
        wspace=0.31, hspace=0.31,
    )
    ax_a = fig.add_subplot(grid[0, :])
    ax_b = fig.add_subplot(grid[1, 0])
    ax_c = fig.add_subplot(grid[1, 1])
    ax_d = fig.add_subplot(grid[2, 0])
    ax_e = fig.add_subplot(grid[2, 1])

    # a, global evidence map with the study design embedded at right.
    ax_a.set_axis_off()
    title_artist = ax_a.text(0.0, 1.025, "(a) Global evidence and experiment design",
                             transform=ax_a.transAxes, ha="left", va="bottom",
                             fontsize=12, fontweight="bold", color="black",
                             clip_on=False, zorder=20)
    title_artist.set_gid("thermal-heterogeneity-panel-title")
    map_axis = ax_a.inset_axes([0.00, 0.235, 0.735, 0.735])
    legend_axis = ax_a.inset_axes([0.00, 0.005, 1.00, 0.180])
    flow_axis = ax_a.inset_axes([0.755, 0.235, 0.235, 0.735])
    legend_axis.set_axis_off()
    flow_axis.set_axis_off()

    draw_world_land(
        map_axis, facecolor="#EFF1EF", edgecolor="#202020", linewidth=0.34
    )
    record_cmap = LinearSegmentedColormap.from_list(
        "record_support", ["#F4E8C8", "#DDB889", "#CF876D", "#A94C43"]
    )
    record_norm = LogNorm(
        vmin=float(country_counts.min()), vmax=float(country_counts.max())
    )
    country_rings = []
    country_colors = []
    for feature in country_asset["features"]:
        key = feature["country_key"]
        if key not in country_counts.index:
            continue
        color = record_cmap(record_norm(float(country_counts.loc[key])))
        for ring in feature["rings"]:
            country_rings.append(np.asarray(ring, dtype=float))
            country_colors.append(color)
    map_axis.add_collection(PolyCollection(
        country_rings, closed=True, facecolors=country_colors,
        edgecolors="#1F1F1F", linewidths=0.32, zorder=1,
    ))

    max_city_records = float(map_points.record_count.max())

    def bubble_area(values):
        values = np.asarray(values, dtype=float)
        return 13 + 92 * values / max_city_records

    for source, marker, face, edge in (
        ("Multinational archive", "o", multinational, multinational_edge),
        ("China archive", "s", "#F4D6A4", china_edge),
    ):
        group = map_points[map_points.source.eq(source)]
        map_axis.scatter(
            group.longitude, group.latitude, s=bubble_area(group.record_count),
            marker=marker, facecolors=face, edgecolors=edge, linewidths=0.55,
            alpha=0.88, zorder=4,
        )

    # Offset weather-context stars from observation markers and retain the true location with a leader.
    star_plot_offsets = {
        "Miami": (-7.0, -7.0), "Singapore": (9.0, -8.0),
        "Delhi": (-9.0, 8.0), "Beijing": (8.0, 8.0),
        "Montreal": (9.0, 7.0), "Malmo": (8.0, 5.0),
        "Tunis": (-8.0, -6.0), "Sydney": (8.0, -7.0),
        "Brasilia": (-10.0, -7.0),
    }
    for number, row in enumerate(climates.itertuples(index=False), start=1):
        delta_lon, delta_lat = star_plot_offsets[row.climate]
        star_lon = row.longitude + delta_lon
        star_lat = row.latitude + delta_lat
        map_axis.plot(
            [row.longitude, star_lon], [row.latitude, star_lat],
            color="#4A4A4A", linewidth=0.55, zorder=5, solid_capstyle="round",
        )
        map_axis.scatter(
            [star_lon], [star_lat], s=112, marker="*", facecolors="white",
            edgecolors="#111111", linewidths=1.0, zorder=6,
        )
        map_axis.annotate(
            str(number), (star_lon, star_lat), xytext=(5, 5),
            textcoords="offset points", ha="left", va="bottom",
            fontsize=11, fontweight="normal", color="black", zorder=7,
            bbox={"boxstyle": "round,pad=0.08", "facecolor": "white",
                  "edgecolor": "none", "alpha": 0.90},
        )

    map_axis.set_xlim(-180, 180)
    map_axis.set_ylim(-55, 82)
    map_axis.set_xticks([-120, -60, 0, 60, 120])
    map_axis.set_yticks([-30, 0, 30, 60])
    map_axis.set_xlabel("")
    map_axis.set_ylabel("")
    map_axis.tick_params(axis="x", labelbottom=False)
    map_axis.tick_params(axis="y", labelsize=11)

    # China inset in an observation-free portion of the South Atlantic.
    china_inset = map_axis.inset_axes([0.51, 0.02, 0.25, 0.32])
    draw_world_land(
        china_inset, facecolor="#F7F5F0", edgecolor="#333333", linewidth=0.30
    )
    china_feature = next(
        feature for feature in country_asset["features"]
        if feature["country_key"] == "china"
    )
    china_inset.add_collection(PolyCollection(
        [np.asarray(ring, dtype=float) for ring in china_feature["rings"]],
        closed=True, facecolors=[record_cmap(record_norm(float(country_counts.loc["china"])))],
        edgecolors="#202020", linewidths=0.35, zorder=1,
    ))
    china_points = map_points[map_points.source.eq("China archive")]
    china_inset.scatter(
        china_points.longitude, china_points.latitude,
        s=bubble_area(china_points.record_count), marker="s",
        facecolors="#F4D6A4", edgecolors=china_edge, linewidths=0.50, zorder=4,
    )
    china_inset.set_xlim(103, 129)
    china_inset.set_ylim(20, 50)
    china_inset.set_xticks([])
    china_inset.set_yticks([])
    china_inset.set_xlabel("")
    china_inset.set_ylabel("")
    china_inset.set_title("China archive cities", fontsize=11, fontweight="normal", pad=1.6)
    china_extent = Rectangle(
        (103, 20), 26, 30, fill=False, edgecolor=china_edge, linewidth=0.9,
        linestyle=(0, (3, 2)), zorder=5,
    )
    map_axis.add_patch(china_extent)
    for source_point, inset_point in (((103, 20), (1, 1)), ((129, 20), (1, 0))):
        map_axis.add_artist(ConnectionPatch(
            xyA=source_point, coordsA=map_axis.transData,
            xyB=inset_point, coordsB=china_inset.transAxes,
            color=china_edge, linewidth=0.6, alpha=0.8, zorder=3, clip_on=False,
        ))

    # All map keys use a dedicated band outside the geographic data region.
    legend_axis.set_xlim(0, 1)
    legend_axis.set_ylim(0, 1)
    legend_axis.text(0.000, 0.97, "City source", ha="left", va="top",
                     fontsize=11, fontweight="normal")
    legend_axis.scatter([0.012, 0.012], [0.60, 0.22], s=[46, 46],
                        marker="o", facecolors=[multinational, "none"],
                        edgecolors=[multinational_edge, "none"], linewidths=0.75)
    legend_axis.scatter([0.012], [0.22], s=46, marker="s", facecolors="#F4D6A4",
                        edgecolors=china_edge, linewidths=0.75)
    legend_axis.text(0.027, 0.60, "Multinational", ha="left", va="center", fontsize=11)
    legend_axis.text(0.027, 0.22, "China", ha="left", va="center", fontsize=11)

    legend_axis.text(0.190, 0.97, "City records", ha="left", va="top",
                     fontsize=11, fontweight="normal")
    size_values = [100, 1000, 3000]
    size_x = [0.215, 0.270, 0.330]
    for x_value, record_count in zip(size_x, size_values):
        legend_axis.scatter(
            [x_value], [0.53], s=bubble_area([record_count]), marker="o",
            facecolors="white", edgecolors="#555555", linewidths=0.75,
        )
        city_label = {100: "100", 1000: "1k", 3000: "3k"}[record_count]
        legend_axis.text(x_value, 0.10, city_label, ha="center", va="center", fontsize=11)

    legend_axis.text(0.365, 0.97, "Country records",
                     ha="left", va="top", fontsize=11, fontweight="normal")
    legend_axis.text(0.365, 0.64, "log scale",
                     ha="left", va="top", fontsize=11)
    country_bar_axis = legend_axis.inset_axes([0.365, 0.20, 0.190, 0.25])
    colorbar = fig.colorbar(
        ScalarMappable(norm=record_norm, cmap=record_cmap),
        cax=country_bar_axis, orientation="horizontal",
    )
    colorbar.set_ticks([30, 300, 3000, 10000])
    colorbar.set_ticklabels(["30", "300", "3k", "10k"])
    colorbar.ax.tick_params(labelsize=11, length=2, pad=1)
    colorbar.outline.set_linewidth(0.45)

    legend_axis.scatter([0.585], [0.51], s=72, marker="*", facecolors="white",
                        edgecolors="#111111", linewidths=0.85)
    legend_axis.text(0.600, 0.97, "Weather contexts", ha="left", va="top",
                     fontsize=11, fontweight="normal")
    legend_axis.text(
        0.605, 0.64, "1 Miami · 2 Singapore · 3 Delhi",
        ha="left", va="center", fontsize=11,
    )
    legend_axis.text(
        0.605, 0.36, "4 Beijing · 5 Montreal · 6 Malmö",
        ha="left", va="center", fontsize=11,
    )
    legend_axis.text(
        0.605, 0.08, "7 Tunis · 8 Sydney · 9 Brasilia",
        ha="left", va="center", fontsize=11,
    )

    # Four compact cards show the evidence-to-outcome workflow.
    flow_axis.set_xlim(0, 1)
    flow_axis.set_ylim(0, 1)
    flow_axis.text(
        0.50, 0.985, "Evidence to outcomes",
        ha="center", va="top", fontsize=11, fontweight="normal",
    )
    workflow = [
        (0.735, "Demand evidence", "33,845 records\n15 countries · 2 archives", light_blue, multinational),
        (0.515, "Demand compositions", "Multinational A–C\nChina A–C · 267 each", "#F6EAE5", china),
        (0.295, "Fixed office prototype", "8 climate zones\n9 climates · 3 controls", "#F0EEE6", gold),
        (0.075, "Compared outcomes", "Service · energy · carbon\nFlexibility · effort", "#F1E6E3", adverse),
    ]
    for index, (y, heading, detail, fill, edge) in enumerate(workflow):
        flow_axis.add_patch(FancyBboxPatch(
            (0.025, y), 0.95, 0.17,
            boxstyle="round,pad=0.010,rounding_size=0.018",
            facecolor=fill, edgecolor=edge, linewidth=0.85,
        ))
        flow_axis.text(
            0.055, y + 0.130, heading, ha="left", va="center",
            fontsize=11, fontweight="normal", linespacing=0.86,
        )
        flow_axis.text(
            0.055, y + 0.050, detail, ha="left", va="center",
            fontsize=11, linespacing=0.86,
        )
        if index < len(workflow) - 1:
            flow_axis.annotate(
                "", xy=(0.50, y - 0.035), xytext=(0.50, y - 0.006),
                arrowprops={"arrowstyle": "-|>", "color": "#666666", "lw": 0.75},
            )

    # b, centre and personal range distributions for six fixed compositions.
    ax_b.set_axis_off()
    panel_title(ax_b, "b", "Six demand compositions")
    centre_axis = ax_b.inset_axes([0.26, 0.06, 0.33, 0.82])
    range_axis = ax_b.inset_axes([0.70, 0.06, 0.29, 0.82])
    composition_ids = sorted(compositions.composition_id.unique())
    composition_labels = ["Multinational A", "Multinational B", "Multinational C",
                          "China A", "China B", "China C"]
    source_colors = [multinational] * 3 + [china] * 3

    def draw_ridge(axis, values, y, color, xmin, xmax):
        counts, edges = np.histogram(values, bins=34, range=(xmin, xmax), density=True)
        kernel = np.array([1, 2, 3, 2, 1], dtype=float)
        smooth = np.convolve(counts, kernel / kernel.sum(), mode="same")
        centres = (edges[:-1] + edges[1:]) / 2
        height = 0.34 * smooth / max(float(smooth.max()), 1e-12)
        axis.fill_between(centres, y, y - height, color=color, alpha=0.24, linewidth=0)
        axis.plot(centres, y - height, color=color, linewidth=0.8)
        q10, q25, q50, q75, q90 = np.quantile(values, [0.10, 0.25, 0.50, 0.75, 0.90])
        axis.hlines(y + 0.11, q10, q90, color=color, linewidth=1.0)
        axis.hlines(y + 0.11, q25, q75, color=color, linewidth=3.2)
        axis.scatter(q50, y + 0.11, s=18, color=color, edgecolors="white",
                     linewidths=0.35, zorder=4)

    for index, composition_id in enumerate(composition_ids):
        group = compositions[compositions.composition_id.eq(composition_id)]
        y = 5 - index
        draw_ridge(centre_axis, group.centre_c, y, source_colors[index], 15, 34)
        draw_ridge(range_axis, group.range_width_c, y, source_colors[index], 0, 13)
    for child in (centre_axis, range_axis):
        child.set_ylim(-0.5, 5.75)
        child.set_yticks(range(6), composition_labels[::-1])
        child.spines["top"].set_visible(False)
        child.spines["right"].set_visible(False)
        child.spines["left"].set_visible(False)
        child.tick_params(axis="x", labelsize=11, pad=1)
        child.tick_params(axis="y", length=0, pad=2)
    centre_axis.set_yticklabels(composition_labels[::-1], fontsize=11)
    range_axis.set_yticklabels([])
    centre_axis.set_xlim(15, 34)
    centre_axis.set_xticks([16, 20, 24, 28, 32])
    range_axis.set_xlim(0, 13)
    range_axis.set_xticks([0, 3, 6, 9, 12])
    centre_axis.set_xlabel("Demand centre\n(°C)", fontsize=11, labelpad=1)
    range_axis.set_xlabel("Personal range\nwidth (°C)", fontsize=11, labelpad=1)
    ax_b.text(
        0.61, 0.93, "10–90% line · 25–75% bar · median",
        transform=ax_b.transAxes, ha="center", va="top", fontsize=11,
        color="#4F4F4F",
    )

    # c, variance allocation and leave-one-source robustness.
    panel_title(ax_c, "c", "Variance across scales")
    component_order = ["country", "city", "release_building_group_id",
                       "release_unit_id", "within_unit"]
    component_labels = {
        "country": "Country", "city": "City",
        "release_building_group_id": "Building",
        "release_unit_id": "Within-building", "within_unit": "Repeated",
    }
    component_colors = ["#6F9DB8", "#9FC1CF", "#C7A15B", "#C97861", "#9A8CA2"]
    lookup = variance.set_index("component").share
    left = 0.0
    for component, color in zip(component_order, component_colors):
        value = float(lookup.loc[component]) * 100
        ax_c.barh(4.8, value, left=left, height=0.55, color=color,
                  edgecolor="white", linewidth=0.45, zorder=2)
        centre = left + value / 2
        if component in {"country", "release_unit_id"}:
            ax_c.text(
                centre, 4.8, f"{value:.1f}%", ha="center", va="center",
                fontsize=11, fontweight="normal",
            )
        elif component == "city":
            ax_c.annotate(
                f"{value:.1f}%", xy=(centre, 5.07), xytext=(centre - 2.0, 5.40),
                ha="center", va="bottom", fontsize=11,
                arrowprops={"arrowstyle": "-", "color": "#555555", "lw": 0.55},
            )
        elif component == "release_building_group_id":
            ax_c.annotate(
                f"{value:.1f}%", xy=(centre, 4.53), xytext=(centre + 2.5, 4.28),
                ha="center", va="top", fontsize=11,
                arrowprops={"arrowstyle": "-", "color": "#555555", "lw": 0.55},
            )
        else:
            ax_c.annotate(
                f"{value:.1f}%", xy=(centre, 5.07), xytext=(centre, 5.38),
                ha="center", va="bottom", fontsize=11,
                arrowprops={"arrowstyle": "-", "color": "#555555", "lw": 0.55},
            )
        left += value
    robust_labels = {
        "between_country": "Country", "between_city_within_country": "City",
        "between_building_within_city": "Building",
        "between_analysis_unit_within_building": "Within-building",
        "within_analysis_unit": "Repeated\nobservations",
    }
    robust_order = ["between_country", "between_city_within_country",
                    "between_building_within_city", "between_analysis_unit_within_building",
                    "within_analysis_unit"]
    for row_index, component in enumerate(robust_order):
        group = variance_robust[variance_robust.component.eq(component)]
        y = 3.55 - row_index * 0.72
        minimum = 100 * float(group.loo_min_share.min())
        maximum = 100 * float(group.loo_max_share.max())
        median = 100 * float(group.loo_median_share.median())
        color = component_colors[row_index]
        ax_c.hlines(y, minimum, maximum, color=color, linewidth=2.4, alpha=0.75)
        ax_c.scatter(median, y, s=30, color=color, edgecolors="white",
                     linewidths=0.45, zorder=3)
        ax_c.text(-7.5, y, robust_labels[component], ha="right", va="center",
                  fontsize=11, linespacing=0.78)
    ax_c.set_xlim(-42, 106)
    ax_c.set_ylim(-0.90, 5.90)
    ax_c.set_xticks([0, 25, 50, 75, 100])
    ax_c.set_yticks([])
    ax_c.set_xlabel("Variance share (%)", fontsize=11, labelpad=2)
    ax_c.text(
        103, -0.43, "12 specifications\n27 source removals",
        ha="right", va="center", fontsize=11, linespacing=0.86,
        color="#4F4F4F",
    )
    clean_axis(ax_c, grid=True)

    # d, positions relative to external policy and design ranges.
    panel_title(ax_d, "d", "Policy-range position")
    policy_order = list(dict.fromkeys(policies.policy))
    abbreviations = {
        "Australia model COP": "Australia", "Brazil NR-17": "Brazil",
        "Canada CCOHS": "Canada", "India ECSBC 2024": "India",
        "Indonesia Permenkes 48": "Indonesia", "Malaysia ICOP IAQ": "Malaysia",
        "Singapore SS 553": "Singapore", "Sweden AFS 2023:12": "Sweden",
        "USA OSHA guidance": "United States",
    }
    range_colors = {"below": "#7FA9C1", "within": "#84AD99", "above": "#D28C76"}
    for y, policy in enumerate(policy_order):
        group = policies[policies.policy.eq(policy)].set_index("position")
        left = 0.0
        for position in ("below", "within", "above"):
            value = 100 * float(group.loc[position, "proportion"])
            ax_d.barh(y, value, left=left, height=0.64, color=range_colors[position],
                      edgecolor="white", linewidth=0.4)
            if position == "within" and value >= 9:
                ax_d.text(left + value / 2, y, f"{value:.0f}%", ha="center", va="center",
                          fontsize=11, color="black")
            left += value
        ax_d.text(102, y, f"n={int(group.loc['within', 'records']):,}",
                  ha="left", va="center", fontsize=11)
    ax_d.set_yticks(
        range(len(policy_order)), [abbreviations.get(p, p) for p in policy_order],
        fontsize=11,
    )
    ax_d.invert_yaxis()
    ax_d.set_xlim(0, 132)
    ax_d.set_xticks([0, 25, 50, 75, 100])
    ax_d.tick_params(axis="x", labelsize=11)
    ax_d.set_xlabel("Demand records (%)", fontsize=11, labelpad=2)
    ax_d.set_ylim(len(policy_order) - 0.5, -2.30)
    ax_d.legend(
        handles=[Patch(facecolor=range_colors[p], label=p.title())
                 for p in ("below", "within", "above")],
        loc="upper center", bbox_to_anchor=(0.50, 0.975), ncol=3, frameon=False, facecolor="none", edgecolor="none",
        fontsize=11, handlelength=0.9, columnspacing=0.8, handletextpad=0.3,
    )
    clean_axis(ax_d, grid=False)

    # e, direct shared-room request disagreement.
    panel_title(ax_e, "e", "Shared-room requests")
    category_order = ["Same", "Mixed", "Opposed"]
    category_colors = {"Same": pale_grey, "Mixed": gold, "Opposed": adverse}
    dataset_order = ["NUS_HAAS", "ENTH"]
    for y, dataset in enumerate(dataset_order):
        group = rooms[rooms.dataset.eq(dataset)].set_index("category")
        left = 0.0
        total = int(group.eligible_windows.iloc[0])
        for category in category_order:
            count = int(group.loc[category, "window_count"])
            share = 100 * count / total
            ax_e.barh(y, share, left=left, height=0.55, color=category_colors[category],
                      edgecolor="white", linewidth=0.5)
            if share >= 9 or category == "Opposed":
                ax_e.text(
                    left + share / 2, y, str(count), ha="center", va="center",
                    fontsize=11, fontweight="normal",
                )
            left += share
    ax_e.set_yticks(
        range(2), ["Singapore\n(n=63)", "Longitudinal\n(n=34)"], fontsize=11,
    )
    ax_e.invert_yaxis()
    ax_e.set_xlim(0, 100)
    ax_e.set_xticks([0, 25, 50, 75, 100])
    ax_e.tick_params(axis="x", labelsize=11)
    ax_e.set_xlabel("Room-windows (%)", fontsize=11, labelpad=2)
    ax_e.set_ylim(1.65, -1.00)
    ax_e.legend(
        handles=[Patch(facecolor=category_colors[c], label=c) for c in category_order],
        loc="upper center", bbox_to_anchor=(0.50, 0.985), ncol=3, frameon=False, facecolor="none", edgecolor="none",
        fontsize=11, handlelength=0.9, columnspacing=0.8, handletextpad=0.3,
    )
    clean_axis(ax_e, grid=False)

    caption = (
        "Figure 1 | Observed thermal demand and the global experiment design. "
        "a, Country colour gives the logarithmically scaled number of observation records for "
        "observed countries represented in the 1:110m base map. City markers retain all observation "
        "locations; marker area scales with record count, and marker shape and edge distinguish the two archives. "
        "The outlined China extent connects to the enlarged city inset. Numbered stars mark the nine "
        "weather contexts paired with one fixed office prototype. The right-hand workflow links the two "
        "harmonized archives to six demand compositions, three controls and "
        "the compared outcomes. b, Demand-centre and estimated personal range width distributions "
        "in the six fixed 267-profile compositions. c, Unit-weighted hierarchical variance shares "
        "for multinational-archive records with available building identities. The stacked bar "
        "shows five components; repeated observations denote variation within an analysis unit. "
        "Points and lines show ranges across 12 specifications and 27 "
        "leave-one-source removals. d, Shares of reconstructed demand below, within and above nine "
        "external policy or design ranges. e, Mutually exclusive request categories in two direct "
        "shared-room datasets; Mixed contains different requests without simultaneous warmer and "
        "cooler requests, whereas Opposed contains both."
    )
    caption_path = RESULTS / "captions" / f"{LABEL}.txt"
    caption_path.parent.mkdir(parents=True, exist_ok=True)
    caption_path.write_text(caption, encoding="utf-8")

    tiff_path = PLOT / "main" / f"{LABEL}.tiff"
    tiff_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(tiff_path, dpi=600, pil_kwargs={"compression": "tiff_lzw"})
    save_plot(fig, LABEL, dpi=600, size_mm=(180, 230), exact_pixels=True)


draw_figure_01()


In [ ]:
# Figure_02: averaging changes plan qualification and thermal-service assessment.
def draw_figure_02():
    from matplotlib.colors import ListedColormap

    story_root = RESULTS / "global_extension/final_story"
    info = pd.read_csv(story_root / "FINAL_INFORMATION_DIAGNOSTIC.csv")
    direct = pd.read_csv(story_root / "FINAL_DIRECT_COMPARISONS.csv")
    summary = json.loads((story_root / "FINAL_STORY_SUMMARY.json").read_text(encoding="utf-8"))
    info = info.loc[np.isclose(info["service_fraction"], 1.0)].copy()
    direct["demand_source"] = np.where(
        direct["composition"].str.startswith("China"), "China archive", "Multinational archive"
    )

    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 210/25.4))
    fig.subplots_adjust(left=0.13, right=0.985, bottom=0.072, top=0.885,
                        hspace=0.80, wspace=0.50)

    # a, cell-by-cell diagnosis of what is lost when the distribution is averaged.
    ax = axes[0, 0]
    final_title(ax, "a", "Averages can select plans\nthat miss individual limits")
    climates = list(CONTEXT_ORDER)
    compositions = ["Multinational A", "Multinational B", "Multinational C",
                    "China A", "China B", "China C"]
    diagnostic = np.zeros((len(climates), len(compositions)), dtype=int)
    for row_index, climate in enumerate(climates):
        for col_index, composition in enumerate(compositions):
            row = info[(info["climate"] == climate) & (info["composition"] == composition)].iloc[0]
            if bool(row["zone_mean_selected_plan_fails_full_review"]):
                diagnostic[row_index, col_index] = 2
            elif bool(row["eligibility_disagreement"]):
                diagnostic[row_index, col_index] = 1
    cmap = ListedColormap(["#EEF0EF", "#F2D7C5", "#8066A8"])
    ax.imshow(diagnostic, cmap=cmap, vmin=-0.5, vmax=2.5, aspect="auto")
    ax.set_xticks(range(len(compositions)), ["A", "B", "C", "A", "B", "C"], fontsize=9)
    ax.text(1.0, -0.16, "Multinational archive", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8.5)
    ax.text(4.0, -0.16, "China archive", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8.5)
    ax.set_yticks(range(len(climates)), [CONTEXT_LABELS[name] for name in climates], fontsize=9)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#B9B9B6")
    outcome_handles = [
        Patch(facecolor="#EEF0EF", edgecolor="#B9B9B6", label="Eligible plans unchanged"),
        Patch(facecolor="#F2D7C5", edgecolor="#B9B9B6", label="Eligible plans changed"),
        Patch(facecolor="#8066A8", edgecolor="#8066A8", label="Mean-selected plan misses full-profile limits"),
    ]
    ax.legend(handles=outcome_handles, loc="upper left", bbox_to_anchor=(-0.035, -0.21),
              ncol=1, borderaxespad=0, frameon=False, facecolor="none",
              edgecolor="none", handlelength=1.0, labelspacing=0.35)

    # b, physical context of selection failures.
    ax = axes[0, 1]
    final_title(ax, "b", "Missed limits cluster in\nheating-led cases")
    failure = info["zone_mean_selected_plan_fails_full_review"].astype(bool)
    groups = [("Other cases", info.loc[~failure, "heating_share"] * 100),
              ("Mean-selected\nplan misses limits", info.loc[failure, "heating_share"] * 100)]
    colors = ["#AAB8C0", ACCENT_PURPLE]
    for index, ((label, values), color) in enumerate(zip(groups, colors)):
        x = index + deterministic_jitter(len(values), 0.13)
        ax.scatter(x, values, s=24, color=color, alpha=0.78, edgecolor="white", linewidth=0.45, zorder=3)
        add_median_iqr(ax, index, values, color)
        ax.text(index, 97, f"n = {len(values)}", ha="center", va="top", fontsize=9)
    ax.set_xlim(-0.55, 1.55); ax.set_ylim(-2, 100)
    ax.set_xticks([0, 1], [item[0] for item in groups])
    ax.set_ylabel("Space-heating share (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    # c-d, service changes in the 108 direct comparisons.
    for ax, metric, title, ylabel, letter in [
        (axes[1, 0], "delta_mean_reference_mismatch_c",
         "Population mismatch", "Reduction in population mismatch (°C)", "c"),
        (axes[1, 1], "delta_fixed_tail_reference_mismatch_c",
         "Least-served mismatch", "Reduction in least-served mismatch (°C)", "d"),
    ]:
        final_title(ax, letter, title)
        for index, control in enumerate(("personal_center", "personal_range")):
            values = -direct.loc[direct["alternative_control"] == control, metric]
            sources = direct.loc[direct["alternative_control"] == control, "demand_source"]
            for source in SOURCE_MARKERS:
                chosen = sources == source
                y = values.loc[chosen].to_numpy()
                x = index + deterministic_jitter(len(y), 0.15)
                ax.scatter(x, y, s=24, marker=SOURCE_MARKERS[source],
                           color=CONTROL_COLORS[control], alpha=0.72,
                           edgecolor="white", linewidth=0.45, zorder=3)
            add_median_iqr(ax, index, values, CONTROL_COLORS[control])
        ax.set_xticks([0, 1], ["Personal\ncentre", "Personal\nrange"])
        ax.set_xlim(-0.5, 1.5); ax.set_ylabel(ylabel, labelpad=4)
        final_style(ax, zero=True, grid_axis="y")

    # e, robustness to three independent analytical choices.
    ax = axes[2, 0]
    final_title(ax, "e", "Sensitivity of missed limits")
    robustness_rows = []
    tail = pd.read_csv(RESULTS / "global_extension/tail_fraction_sensitivity/TAIL_FRACTION_INFORMATION_DIAGNOSTIC.csv")
    tail = tail.loc[np.isclose(tail["service_fraction"], 1.0)]
    for fraction, group in tail.groupby("tail_fraction"):
        resolvable = group.loc[~group["full_unresolved"].astype(bool)]
        robustness_rows.append((f"Tail {int(round(fraction*100))}%", int(resolvable["zone_mean_selected_plan_fails_full_review"].sum()), len(resolvable), "Tail group"))
    anchors = pd.read_csv(RESULTS / "global_extension/reference_anchor_sensitivity/REFERENCE_ANCHOR_HEADLINE.csv")
    for row in anchors.itertuples(index=False):
        robustness_rows.append((f"Anchor {row.reference_target_c:.0f}°C", int(row.zone_mean_selected_plan_fails_full_review_at_fraction_1), int(row.full_resolvable_scenarios_at_fraction_1), "Reference"))
    for directory, label, scope in [("target_domain_final", "20–27°C", "declared_20_27_c"),
                                     ("target_domain_20_26", "20–26°C", "central_20_26_c"),
                                     ("target_domain_21_27", "21–27°C", "central_21_27_c"),
                                     ("target_domain_21_26", "21–26°C", "central_21_26_c")]:
        domain = pd.read_csv(RESULTS / f"global_extension/{directory}/TARGET_DOMAIN_INFORMATION_DIAGNOSTIC.csv")
        domain = domain.loc[np.isclose(domain["service_fraction"], 1.0) & domain["domain_scope"].eq(scope)]
        resolvable = domain.loc[~domain["full_unresolved"].astype(bool)]
        robustness_rows.append((label, int(resolvable["zone_mean_selected_plan_fails_full_review"].sum()), len(resolvable), "Target domain"))
    robustness = pd.DataFrame(robustness_rows, columns=["setting", "failures", "denominator", "family"])
    robustness["rate"] = robustness["failures"] / robustness["denominator"] * 100
    family_colors = {"Tail group": "#4C82B8", "Reference": ACCENT_ORANGE, "Target domain": ACCENT_PURPLE}
    y = np.arange(len(robustness))[::-1]
    ax.hlines(y, 0, robustness["rate"], color="#D4D4D1", linewidth=2, zorder=1)
    for family, group in robustness.groupby("family", sort=False):
        positions = y[group.index]
        ax.scatter(group["rate"], positions, s=38, color=family_colors[family], label=family, zorder=3)
    for yy, row in zip(y, robustness.itertuples(index=False)):
        ax.text(row.rate + 0.7, yy, f"{row.failures}/{row.denominator}", va="center", fontsize=8.3)
    ax.set_yticks(y, robustness["setting"], fontsize=8.5)
    ax.set_xlim(0, max(30, robustness["rate"].max()+7))
    ax.set_xlabel("Mean-selected plan misses full-profile\nlimits (%)")
    final_style(ax, grid_axis="x")

    # f, three complementary service tests.
    ax = axes[2, 1]
    final_title(ax, "f", "Three service outcomes")
    service_metrics = [
        ("Population\nmismatch", "delta_mean_reference_mismatch_c", "lower"),
        ("Least\nserved\nmismatch", "delta_fixed_tail_reference_mismatch_c", "lower"),
        ("Personal\nrange\ncoverage", "delta_modeled_interval_person_hour_coverage", "higher"),
    ]
    base_x = np.arange(len(service_metrics))
    width = 0.32
    for offset, control in [(-width/2, "personal_center"), (width/2, "personal_range")]:
        subset = direct.loc[direct["alternative_control"] == control]
        rates = []
        counts = []
        for _, metric, direction in service_metrics:
            count = int((subset[metric] < 0).sum()) if direction == "lower" else int((subset[metric] > 0).sum())
            counts.append(count); rates.append(count / len(subset) * 100)
        bars = ax.bar(base_x + offset, rates, width=width, color=CONTROL_COLORS[control],
                      alpha=0.88, label=CONTROL_LABELS[control], zorder=3)
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.2, f"{count}/54",
                    ha="center", va="bottom", fontsize=8.4)
    ax.set_xticks(base_x, [item[0] for item in service_metrics], fontsize=8.8)
    ax.set_ylim(0, 112); ax.set_ylabel("Comparisons improved (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    control_handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                              color=CONTROL_COLORS[c], label=CONTROL_LABELS[c])
                       for c in ("personal_center", "personal_range")]
    source_handles = [Line2D([0], [0], marker=SOURCE_MARKERS[s], linestyle="", markersize=6,
                             markerfacecolor="#73777A", markeredgecolor="white", label=s)
                      for s in SOURCE_MARKERS]
    fig.legend(handles=control_handles + source_handles, loc="upper center", ncol=4,
               bbox_to_anchor=(0.5, 0.982), frameon=False, facecolor="none", edgecolor="none", columnspacing=1.6, handletextpad=0.5)
    save_plot(fig, "Figure_02", size_mm=(180, 210))

draw_figure_02()


In [ ]:
# Figure_03: service, whole-building energy, carbon and seasonal shifts.
def draw_figure_03():
    story_root = RESULTS / "global_extension/final_story"
    direct = pd.read_csv(story_root / "FINAL_DIRECT_COMPARISONS.csv")
    matched = pd.read_csv(story_root / "FINAL_MATCHED_SERVICE.csv")
    seasonal = pd.read_csv(story_root / "FINAL_SEASONAL_REALLOCATION.csv")
    summary = json.loads((story_root / "FINAL_STORY_SUMMARY.json").read_text(encoding="utf-8"))
    direct["demand_source"] = np.where(
        direct["composition"].str.startswith("China"), "China archive", "Multinational archive"
    )

    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 210/25.4))
    fig.subplots_adjust(left=0.16, right=0.970, bottom=0.072, top=0.875,
                        hspace=0.88, wspace=0.55)

    # a, direct operation: service against whole-building energy.
    ax = axes[0, 0]
    final_title(ax, "a", "Service gains usually raise energy")
    for (control, source), group in direct.groupby(["alternative_control", "demand_source"]):
        ax.scatter(-group["delta_mean_reference_mismatch_c"], group["delta_facility_purchased_pct"],
                   s=27, marker=SOURCE_MARKERS[source], color=CONTROL_COLORS[control],
                   alpha=0.72, edgecolor="white", linewidth=0.45, zorder=3)
    ax.axvline(0, color="#AAA9A5", linewidth=0.8, zorder=0)
    ax.set_xlabel("Reduction in population mismatch (°C)")
    ax.set_ylabel("Whole-building energy change (%)", labelpad=4)
    final_style(ax, zero=True, grid_axis="y")

    # b, country-specific carbon factors preserve the energy direction but alter magnitude.
    ax = axes[0, 1]
    final_title(ax, "b", "Energy change\ndetermines carbon")
    for (control, source), group in direct.groupby(["alternative_control", "demand_source"]):
        ax.scatter(group["delta_facility_purchased_kwh"] / 1000,
                   group["delta_operational_co2_country_kg"] / 1000,
                   s=27, marker=SOURCE_MARKERS[source], color=CONTROL_COLORS[control],
                   alpha=0.72, edgecolor="white", linewidth=0.45, zorder=3)
    ax.axvline(0, color="#AAA9A5", linewidth=0.8, zorder=0)
    ax.set_xlim(-9, 46); ax.set_xticks([0, 20, 40])
    ax.set_xlabel("Whole-building energy change (MWh)")
    ax.set_ylabel("Operational carbon change (t CO$_2$e)", labelpad=4)
    final_style(ax, zero=True, grid_axis="y")

    # c, four complementary comparison scopes.
    ax = axes[1, 0]
    final_title(ax, "c", "Energy direction persists")
    direct_direction = summary["direct_service_and_resources"]["all_personal_controls"]["facility_energy_direction"]
    common_direction = summary["energy_boundary_sensitivity"]["service_selected_energy_directions"]["complete_facility"]["1"]
    near = matched.loc[matched["within_tolerance_0p100_c"].astype(bool)]
    seasonal_direction = summary["seasonal_reallocation"]["annual_complete_facility_energy_direction"]
    direction_rows = [
        ("Direct operation", int(direct_direction["decrease"]), int(direct_direction["increase"])),
        ("Common service", int(common_direction["decrease"]), int(common_direction["increase"])),
        ("Service gap\n≤0.10°C", int((near["energy_direction"] == "decrease").sum()), int((near["energy_direction"] == "increase").sum())),
        ("Seasonal demand", int(seasonal_direction["decrease"]), int(seasonal_direction["increase"])),
    ]
    y = np.arange(len(direction_rows))[::-1]
    for yy, (label, decrease, increase) in zip(y, direction_rows):
        total = decrease + increase
        decrease_share, increase_share = decrease / total * 100, increase / total * 100
        ax.barh(yy, -decrease_share, color="#4C82B8", height=0.56, zorder=3)
        ax.barh(yy, increase_share, color=ACCENT_ORANGE, height=0.56, zorder=3)
        ax.text(-decrease_share/2, yy, str(decrease), ha="center", va="center", fontsize=8.6,
                color="white" if decrease_share > 12 else "black").set_gid("preserve-text-color" if decrease_share > 12 else None)
        ax.text(increase_share/2, yy, str(increase), ha="center", va="center", fontsize=8.6,
                color="white" if increase_share > 16 else "black").set_gid("preserve-text-color" if increase_share > 16 else None)
        ax.text(101.5, yy, f"n={total}", ha="left", va="center", fontsize=8.4)
    ax.set_yticks(y, [row[0] for row in direction_rows], fontsize=9)
    ax.set_xlim(-28, 112); ax.set_xlabel("Share of comparisons (%)")
    ax.axvline(0, color="#777777", linewidth=0.9)
    ax.set_ylim(-0.55, 4.05)
    ax.text(-14, 3.72, "Energy lower", ha="center", va="center", fontsize=8.6, color="#4C82B8")
    ax.text(43, 3.72, "Energy higher", ha="center", va="center", fontsize=8.6, color=ACCENT_ORANGE)
    final_style(ax, grid_axis="x")

    # d, paired operations whose realised service is progressively more similar.
    ax = axes[1, 1]
    final_title(ax, "d", "Matched service\nretains an energy gap")
    tolerances = [0.010, 0.025, 0.050, 0.075, 0.100]
    lower_counts, higher_counts = [], []
    for tolerance in tolerances:
        column = "within_tolerance_" + f"{tolerance:.3f}".replace("0.", "0p") + "_c"
        subset = matched.loc[matched[column].astype(bool)]
        lower_counts.append(int((subset["energy_direction"] == "decrease").sum()))
        higher_counts.append(int((subset["energy_direction"] == "increase").sum()))
    x = np.arange(len(tolerances))
    ax.bar(x, lower_counts, width=0.68, color="#4C82B8", label="Energy lower", zorder=3)
    ax.bar(x, higher_counts, width=0.68, bottom=lower_counts, color=ACCENT_ORANGE,
           label="Energy higher", zorder=3)
    for xx, lower, higher in zip(x, lower_counts, higher_counts):
        total = lower + higher
        ax.text(xx, total + 2.0, f"n={total}", ha="center", va="bottom", fontsize=8.5)
        if lower:
            ax.text(xx, lower/2, str(lower), ha="center", va="center", fontsize=8.2, color="white").set_gid("preserve-text-color")
    ax.set_xticks(x, [f"{value:.3f}".rstrip("0") for value in tolerances])
    ax.set_xlabel("Maximum service gap (°C)")
    ax.set_ylabel("Matched comparisons", labelpad=4)
    ax.set_ylim(0, max(np.add(lower_counts, higher_counts)) + 42)
    direction_handles = [Patch(facecolor="#4C82B8", label="Energy lower"),
                         Patch(facecolor=ACCENT_ORANGE, label="Energy higher")]
    ax.legend(handles=direction_handles, loc="upper center", ncol=2, frameon=False,
              facecolor="none", edgecolor="none", fontsize=8.4)
    final_style(ax, grid_axis="y")

    # e, seasonal demand profiles assessed over the year.
    ax = axes[2, 0]
    final_title(ax, "e", "Seasonal shifts retain\nthe annual energy gap")
    for (control, source, contrast), group in seasonal.groupby(
        ["alternative_control", "demand_source", "seasonal_contrast_fraction"]
    ):
        face = CONTROL_COLORS[control] if np.isclose(contrast, 1.0) else "none"
        ax.scatter(-group["delta_annual_mean_reference_mismatch_c"],
                   group["delta_annual_facility_purchased_pct"], s=27,
                   marker=SOURCE_MARKERS[source], facecolor=face,
                   edgecolor=CONTROL_COLORS[control], linewidth=0.8, alpha=0.72, zorder=3)
    ax.axvline(0, color="#AAA9A5", linewidth=0.8, zorder=0)
    ax.set_xlabel("Reduction in annual population mismatch (°C)")
    ax.set_ylabel("Whole-building energy change (%)", labelpad=4)
    contrast_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="none", markeredgecolor="#73777A", label="Half contrast"),
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="#73777A", markeredgecolor="#73777A", label="Full contrast"),
    ]
    lower, upper = ax.get_ylim()
    ax.set_ylim(lower, max(upper, 16.0))
    ax.legend(handles=contrast_handles, loc="upper left", ncol=2, frameon=False, facecolor="none", edgecolor="none", fontsize=8.3)
    final_style(ax, zero=True, grid_axis="y")

    # f, cancellation between positive and negative seasonal shifts.
    ax = axes[2, 1]
    final_title(ax, "f", "Seasonal cancellation is limited")
    labels = []
    for index, contrast in enumerate(sorted(seasonal["seasonal_contrast_fraction"].unique())):
        subset = seasonal.loc[np.isclose(seasonal["seasonal_contrast_fraction"], contrast)]
        values = subset["seasonal_energy_cancellation_fraction"] * 100
        x_values = index + deterministic_jitter(len(values), 0.18)
        color = "#8BB4CB" if contrast < 1 else ACCENT_PURPLE
        ax.scatter(x_values, values, s=23, color=color, alpha=0.64,
                   edgecolor="white", linewidth=0.4, zorder=3)
        add_median_iqr(ax, index, values, color)
        above = int((values > 50).sum())
        mixed = int(subset["mixed_seasonal_energy_signs"].astype(bool).sum())
        ax.text(index, 120, f">50%: {above}/{len(values)}\nMixed: {mixed}/{len(values)}",
                ha="center", va="top", fontsize=8.4)
        labels.append("Half\ncontrast" if contrast < 1 else "Full\ncontrast")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_xlim(-0.55, len(labels)-0.45); ax.set_ylim(-3, 126); ax.set_yticks([0, 50, 100])
    ax.set_ylabel("Seasonal cancellation (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    control_handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                              color=CONTROL_COLORS[c], label=CONTROL_LABELS[c])
                       for c in ("personal_center", "personal_range")]
    source_handles = [Line2D([0], [0], marker=SOURCE_MARKERS[s], linestyle="", markersize=6,
                             markerfacecolor="#73777A", markeredgecolor="white", label=s)
                      for s in SOURCE_MARKERS]
    legend_axis = fig.add_axes([0.08, 0.905, 0.84, 0.055])
    legend_axis.set_gid("thermal-heterogeneity-global-legend-axis")
    legend_axis.axis("off")
    legend_axis.legend(handles=control_handles + source_handles, loc="center", ncol=4,
                       frameon=False, facecolor="none", edgecolor="none",
                       columnspacing=1.6, handletextpad=0.5)
    save_plot(fig, "Figure_03", size_mm=(180, 210))

draw_figure_03()


In [ ]:
# Figure_04: demand placement and command permission are separate implementation levers.
def draw_figure_04():
    spatial = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_SPATIAL_EFFECTS.csv")
    personal = spatial[spatial["control"].isin(["personal_center", "personal_range"])].copy()
    permission = personal[personal["axis"] == "permission"].copy()
    placement = personal[personal["axis"] == "layout"].copy()

    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 210/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.072, top=0.875,
                        hspace=0.88, wspace=0.55)

    # a, visual separation of the two levers.
    ax = axes[0, 0]
    final_title(ax, "a", "Placement and permission act separately")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.text(0.23, 0.91, "Demand arrangement", ha="center", va="center")
    ax.text(0.75, 0.89, "Command scope", ha="center", va="center", fontweight="bold", fontsize=10)
    person_colors = ["#4C82B8", "#D47C4A", "#8066A8"]
    random_order = [0, 1, 2, 2, 0, 1, 1, 2, 0]
    ordered = [0, 0, 0, 1, 1, 1, 2, 2, 2]
    for row, (label, order) in enumerate([("Mixed", random_order), ("Grouped", ordered)]):
        y = 0.59 - row * 0.36
        ax.text(0.20, y+0.14, label, ha="center", va="bottom")
        for index, color_index in enumerate(order):
            x = 0.105 + (index % 3) * 0.085
            yy = y + (1 - index // 3) * 0.055 - 0.035
            ax.scatter(x, yy, s=26, color=person_colors[color_index], edgecolor="white", linewidth=0.4)
        ax.add_patch(Rectangle((0.075, y-0.10), 0.25, 0.20, fill=False, edgecolor="#8A8A87", linewidth=0.8))
    stages = [("Building (1)", 0.64), ("Floor (3)", 0.43), ("Zone (15)", 0.22)]
    for label, y in stages:
        ax.add_patch(FancyBboxPatch((0.60, y-0.075), 0.31, 0.15,
                                    boxstyle="round,pad=0.012,rounding_size=0.02",
                                    facecolor=PALE_BLUE if "Zone" not in label else PALE_GREEN,
                                    edgecolor="#6C7E86", linewidth=0.8))
        ax.text(0.755, y, label, ha="center", va="center")
    ax.annotate("", xy=(0.755, 0.51), xytext=(0.755, 0.56), arrowprops=dict(arrowstyle="->", color="#777777", lw=0.8))
    ax.annotate("", xy=(0.755, 0.30), xytext=(0.755, 0.35), arrowprops=dict(arrowstyle="->", color="#777777", lw=0.8))

    # b-c, permission contrasts.
    transitions = [("building", "floor", "Building→floor"),
                   ("floor", "native_zone", "Floor→zone"),
                   ("building", "native_zone", "Building→zone")]
    for ax, metric, title, ylabel, letter, invert in [
        (axes[0, 1], "delta_mean_reference_mismatch_c", "Service gain from\nmore commands",
         "Reduction in population mismatch (°C)", "b", True),
        (axes[1, 0], "delta_facility_purchased_pct", "More commands change energy",
         "Whole-building energy change (%)", "c", False),
    ]:
        final_title(ax, letter, title)
        positions, labels = [], []
        for transition_index, (reference, alternative, label) in enumerate(transitions):
            for control_index, control in enumerate(("personal_center", "personal_range")):
                subset = permission[(permission["reference"] == reference) &
                                    (permission["alternative"] == alternative) &
                                    (permission["control"] == control)]
                values = subset[metric].copy()
                if invert:
                    values = -values
                position = transition_index * 1.25 + (control_index - 0.5) * 0.32
                positions.append(position)
                jitter = deterministic_jitter(len(values), 0.10)
                ax.scatter(position + jitter, values, s=20, color=CONTROL_COLORS[control],
                           alpha=0.48, edgecolor="white", linewidth=0.35, zorder=3)
                add_median_iqr(ax, position, values, CONTROL_COLORS[control])
            labels.append(label)
        ax.set_xticks(np.arange(len(transitions))*1.25, [item[2].replace("→", "→\n") for item in transitions], fontsize=8.8)
        ax.set_xlim(-0.55, (len(transitions)-1)*1.25+0.55)
        ax.set_ylabel(ylabel, labelpad=4)
        final_style(ax, zero=True, grid_axis="y")

    # d-e, placement contrasts at unchanged command permission.
    layouts = [("demand_ordered", "Demand\nordered"),
               ("demand_ordered_reversed", "Order\nreversed"),
               ("demand_ordered_shift5", "Order\nshifted")]
    for ax, metric, title, ylabel, letter, invert in [
        (axes[1, 1], "delta_mean_reference_mismatch_c", "Service gain from\ncompatible placement",
         "Reduction in population mismatch (°C)", "d", True),
        (axes[2, 0], "delta_facility_purchased_pct", "Placement redistributes energy",
         "Whole-building energy change (%)", "e", False),
    ]:
        final_title(ax, letter, title)
        for layout_index, (layout, label) in enumerate(layouts):
            for control_index, control in enumerate(("personal_center", "personal_range")):
                subset = placement[(placement["alternative"] == layout) & (placement["control"] == control)]
                values = subset[metric].copy()
                if invert:
                    values = -values
                position = layout_index * 1.25 + (control_index - 0.5) * 0.32
                jitter = deterministic_jitter(len(values), 0.10)
                ax.scatter(position + jitter, values, s=20, color=CONTROL_COLORS[control],
                           alpha=0.48, edgecolor="white", linewidth=0.35, zorder=3)
                add_median_iqr(ax, position, values, CONTROL_COLORS[control])
        ax.set_xticks(np.arange(len(layouts))*1.25, [label for _, label in layouts], fontsize=8.8)
        ax.set_xlim(-0.55, (len(layouts)-1)*1.25+0.55)
        ax.set_ylabel(ylabel, labelpad=4)
        final_style(ax, zero=True, grid_axis="y")

    # f, outcome frequencies for the largest permission change.
    ax = axes[2, 1]
    final_title(ax, "f", "Building-to-zone outcomes")
    strongest = permission[(permission["reference"] == "building") &
                           (permission["alternative"] == "native_zone")]
    outcomes = [
        ("Population\nmismatch", "delta_mean_reference_mismatch_c", "lower"),
        ("Least\nserved\ngroup", "delta_fixed_tail_reference_mismatch_c", "lower"),
        ("Range\ncoverage", "delta_modeled_interval_person_hour_coverage", "higher"),
        ("Energy\nincreased", "delta_facility_purchased_pct", "higher"),
    ]
    x = np.arange(len(outcomes)); width = 0.32
    for offset, control in [(-width/2, "personal_center"), (width/2, "personal_range")]:
        subset = strongest[strongest["control"] == control]
        rates, counts = [], []
        for _, metric, direction in outcomes:
            count = int((subset[metric] < 0).sum()) if direction == "lower" else int((subset[metric] > 0).sum())
            counts.append(count); rates.append(count/len(subset)*100)
        bars = ax.bar(x+offset, rates, width=width, color=CONTROL_COLORS[control], zorder=3)
        for bar, count in zip(bars, counts):
            if offset < 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f"{count}/54",
                        ha="center", va="bottom", fontsize=7.9, rotation=0)
            else:
                label = ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()-3, f"{count}/\n54",
                                ha="center", va="top", fontsize=7.9, color="white", rotation=0)
                label.set_gid("preserve-text-color")
    ax.set_xticks(x, [row[0] for row in outcomes], fontsize=8.2)
    ax.set_ylim(0, 115); ax.set_ylabel("Comparisons (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      color=CONTROL_COLORS[c], label=CONTROL_LABELS[c])
               for c in ("personal_center", "personal_range")]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 0.982),
               frameon=False, facecolor="none", edgecolor="none", columnspacing=2.2)
    save_plot(fig, "Figure_04", size_mm=(180, 210))

draw_figure_04()


In [ ]:
# Figure_05: service-qualified response across 72 seasonal event definitions.
def draw_figure_05():
    event = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_EVENT_RESPONSE.csv")
    summary_root = RESULTS / "global_extension/multi_event_dr_final/summary"
    grouped = pd.read_csv(summary_root / "EVENT_RESPONSE_GROUP_SUMMARY.csv")
    paired = pd.read_csv(summary_root / "EVENT_RESPONSE_PAIRED_CONTROL_SUMMARY.csv")
    terminal = pd.read_csv(summary_root / "EVENT_TERMINAL_RECOVERY_GROUP_SUMMARY.csv")
    response_summary = json.loads((summary_root / "EVENT_RESPONSE_SUMMARY.json").read_text(encoding="utf-8"))
    main_factor = float(response_summary["main_service_factor"])
    event = event.loc[np.isclose(event["event_service_limit_factor"], main_factor)].copy()
    grouped = grouped.loc[np.isclose(grouped["event_service_limit_factor"], main_factor)].copy()
    paired = paired.loc[np.isclose(paired["main_service_factor"], main_factor)].copy()

    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 215/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.070, top=0.855,
                        hspace=0.96, wspace=0.58)
    event_color, full_color = "#4C82B8", ACCENT_PURPLE

    # a, audit design and timeline.
    ax = axes[0, 0]
    final_title(ax, "a", "Each control has its own baseline")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    event_definitions = event[["climate", "local_season", "load_level"]].drop_duplicates().shape[0]
    climates = event["climate"].nunique(); seasons = event["local_season"].nunique()
    load_days = event["load_level"].nunique(); demand_groups = event["composition_id"].nunique()
    actions = event["action_id"].nunique(); controls = event["control"].nunique()
    ax.text(0.02, 0.76, f"{climates} climates × {seasons} seasons × {load_days} load days = {event_definitions} events",
            ha="left", va="center", fontweight="bold", fontsize=9.0)
    ax.text(0.02, 0.60, f"{demand_groups} demand groups × {controls} controls × {actions} actions",
            ha="left", va="center")
    ax.text(0.02, 0.49, f"Service limit factor {main_factor:.2f}",
            ha="left", va="center")
    stages = [("Pre-\nevent", 0.02, 0.20, PALE_BLUE),
              ("2-h\nevent", 0.27, 0.16, PALE_ORANGE),
              ("48-h\nrecovery", 0.49, 0.22, PALE_GREEN),
              ("End-state\ncheck", 0.76, 0.22, PALE_PURPLE)]
    for label, left, width, color in stages:
        ax.add_patch(FancyBboxPatch((left, 0.14), width, 0.20,
                                    boxstyle="round,pad=0.01,rounding_size=0.025",
                                    facecolor=color, edgecolor="#7A7A77", linewidth=0.8))
        ax.text(left+width/2, 0.24, label, ha="center", va="center", linespacing=0.95)
    for start, end in [(0.22, 0.27), (0.43, 0.49), (0.71, 0.76)]:
        ax.annotate("", xy=(end, 0.24), xytext=(start, 0.24),
                    arrowprops=dict(arrowstyle="->", lw=0.8, color="#777777"))


    # b, strict-factor qualification by control.
    ax = axes[0, 1]
    final_title(ax, "b", "Service limits narrow\nthe usable response set")
    control_summary = grouped[grouped["group_dimension"] == "control"].set_index("group_value")
    x = np.arange(3); width = 0.34
    event_rates = [control_summary.loc[c, "qualified_positive_event_fraction"]*100 for c in CONTROL_ORDER]
    full_rates = [control_summary.loc[c, "qualified_positive_full_window_fraction"]*100 for c in CONTROL_ORDER]
    bars_event = ax.bar(x-width/2, event_rates, width=width, color=event_color, zorder=3)
    bars_full = ax.bar(x+width/2, full_rates, width=width, color=full_color, zorder=3)
    for bars in (bars_event, bars_full):
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.0, f"{bar.get_height():.0f}%",
                    ha="center", va="bottom", fontsize=8.2)
    ax.set_xticks(x, ["Uniform\naverage", "Personal\ncentre", "Personal\nrange"], fontsize=8.7)
    ax.set_ylim(0, 90); ax.set_ylabel("Qualified positive responses (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    # c, matched qualification gains and losses.
    ax = axes[1, 0]
    final_title(ax, "c", "Demand information\nchanges event eligibility")
    pair_overall = paired[paired["group_dimension"] == "overall"].set_index("alternative_control")
    rows = []
    for control in ("personal_center", "personal_range"):
        row = pair_overall.loc[control]
        rows.extend([
            (("Centre" if control == "personal_center" else "Range")+"\nEvent", int(row["alternative_only_event_qualified"]), int(row["uniform_only_event_qualified"])),
            (("Centre" if control == "personal_center" else "Range")+"\nFull window", int(row["alternative_only_full_window_qualified"]), int(row["uniform_only_full_window_qualified"])),
        ])
    y = np.arange(len(rows))[::-1]
    for yy, (label, guided_only, uniform_only) in zip(y, rows):
        ax.barh(yy, guided_only, color="#56A17A", height=0.54, zorder=3)
        ax.barh(yy, -uniform_only, color=ACCENT_ORANGE, height=0.54, zorder=3)
        guided_label = ax.text(guided_only-8, yy, str(guided_only), ha="right", va="center", color="white")
        guided_label.set_gid("preserve-text-color")
        ax.text(-uniform_only-8, yy, str(uniform_only), ha="right", va="center", fontsize=8.5)
    ax.set_yticks(y, [row[0] for row in rows], fontsize=8.6)
    ax.set_xlim(-95, max(row[1] for row in rows)+75)
    ax.set_xlabel("Matched event pairs")
    ax.axvline(0, color="#777777", linewidth=0.9)
    ax.set_ylim(-0.55, 4.40)
    qualification_handles = [Patch(facecolor=ACCENT_ORANGE, label="Uniform only"),
                             Patch(facecolor="#56A17A", label="Demand-guided only")]
    ax.legend(handles=qualification_handles, loc="upper center", ncol=2,
              frameon=False, facecolor="none", edgecolor="none",
              columnspacing=0.9, handlelength=1.0)
    final_style(ax, grid_axis="x")

    # d, actions expose the recovery trade-off.
    ax = axes[1, 1]
    final_title(ax, "d", "Action design\nshapes full-window value")
    action_labels = {
        "uniform_0.5": "Uniform +0.5°C",
        "uniform_1.0": "Uniform +1.0°C",
        "protected_1.0": "Protected +1.0°C",
        "precondition_0.5_uniform_1.0": "Precondition\n+ uniform",
        "precondition_0.5_protected_1.0": "Precondition\n+ protected",
    }
    action_order = list(action_labels)
    action_summary = grouped[grouped["group_dimension"] == "action_id"].set_index("group_value")
    y = np.arange(len(action_order))[::-1]; height = 0.32
    event_values = [action_summary.loc[item, "qualified_positive_event_fraction"]*100 for item in action_order]
    full_values = [action_summary.loc[item, "qualified_positive_full_window_fraction"]*100 for item in action_order]
    ax.barh(y+height/2, event_values, height=height, color=event_color, zorder=3)
    ax.barh(y-height/2, full_values, height=height, color=full_color, zorder=3)
    for yy, first, second in zip(y, event_values, full_values):
        first_label = ax.text(first-1.2, yy+height/2, f"{first:.0f}", ha="right", va="center", color="white")
        second_label = ax.text(second-1.2, yy-height/2, f"{second:.0f}", ha="right", va="center", color="white")
        first_label.set_gid("preserve-text-color"); second_label.set_gid("preserve-text-color")
    ax.set_yticks(y, [action_labels[item] for item in action_order], fontsize=8.2)
    ax.set_xlim(0, 82); ax.set_xticks([0, 20, 40, 60, 80]); ax.set_xlabel("Qualified positive responses (%)")
    final_style(ax, grid_axis="x")

    # e, four seasons and two load-day definitions.
    ax = axes[2, 0]
    final_title(ax, "e", "Response varies by season")
    season_order = ["winter", "spring", "summer", "autumn"]
    season_summary = grouped[grouped["group_dimension"] == "local_season"].set_index("group_value")
    x = np.arange(4); width = 0.34
    event_values = [season_summary.loc[item, "qualified_positive_event_fraction"]*100 for item in season_order]
    full_values = [season_summary.loc[item, "qualified_positive_full_window_fraction"]*100 for item in season_order]
    bars_event = ax.bar(x-width/2, event_values, width=width, color=event_color, zorder=3)
    bars_full = ax.bar(x+width/2, full_values, width=width, color=full_color, zorder=3)
    for bars in (bars_event, bars_full):
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.0, f"{bar.get_height():.0f}%",
                    ha="center", va="bottom", fontsize=8.2)
    ax.set_xticks(x, [item.title() for item in season_order])
    ax.set_ylim(0, 92); ax.set_ylabel("Qualified positive responses (%)", labelpad=4)
    final_style(ax, grid_axis="y")

    # f, descriptive residuals at the final recovery hour.
    ax = axes[2, 1]
    final_title(ax, "f", "Residuals at the end\nof recovery")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    final_hour = terminal[(terminal["audit_window"] == "final_recovery_hour") &
                          (terminal["group_dimension"] == "overall")].iloc[0]
    cards = [
        ("Operative temperature", float(final_hour["operative_temperature_c_abs_median"]),
         float(final_hour["operative_temperature_c_abs_p95"]), "°C", "#56A17A"),
        ("Whole-building electricity", float(final_hour["facility_electricity_kw_abs_median"]),
         float(final_hour["facility_electricity_kw_abs_p95"]), "kW", ACCENT_PURPLE),
    ]
    for index, (label, median, p95, unit, color) in enumerate(cards):
        y0 = 0.56 - index*0.43
        ax.add_patch(FancyBboxPatch((0.03, y0), 0.94, 0.31,
                                    boxstyle="round,pad=0.015,rounding_size=0.02",
                                    facecolor="#F7F7F5", edgecolor="#C4C4C0", linewidth=0.8))
        ax.add_patch(Rectangle((0.045, y0+0.035), 0.018, 0.24, facecolor=color, edgecolor="none"))
        ax.text(0.09, y0+0.235, label, ha="left", va="center")
        ax.text(0.09, y0+0.085, f"Median\n{median:.3f} {unit}", ha="left", va="center", linespacing=1.0)
        ax.text(0.91, y0+0.085, f"95th percentile\n{p95:.3f} {unit}", ha="right", va="center", linespacing=1.0)
    ax.text(0.5, 0.01, "Absolute residual at recovery hour 48", ha="center", va="bottom")

    response_handles = [Patch(facecolor=event_color, label="Event qualified"),
                        Patch(facecolor=full_color, label="Full-window net-positive")]
    fig.legend(handles=response_handles, loc="upper center", ncol=2,
               bbox_to_anchor=(0.5, 0.982), frameon=False, facecolor="none", edgecolor="none", columnspacing=2.2)
    save_plot(fig, "Figure_05", size_mm=(180, 215))

draw_figure_05()


In [ ]:
# Figure_06: implementation resources and an evidence-based decision frame.
def draw_figure_06():
    root = RESULTS / "global_extension/implementation_value_final"
    boundaries = pd.read_csv(root / "GLOBAL_SERVICE_RESOURCE_VALUE_BOUNDARIES.csv")
    spatial = pd.read_csv(root / "GLOBAL_SPATIAL_IMPLEMENTATION_BOUNDARIES.csv")
    scenarios = pd.read_csv(root / "IMPLEMENTATION_RESOURCE_SCENARIOS.csv")
    catalog = pd.read_csv(root / "IMPLEMENTATION_RESOURCE_CATALOG.csv")
    commands = pd.read_csv(root / "COMMAND_SCOPE_REQUIREMENTS.csv")
    information = pd.read_csv(root / "INFORMATION_BURDEN_SUMMARY.csv").iloc[0]
    compression = json.loads((RESULTS / "global_extension/information_compression_final/GLOBAL_INFORMATION_COMPRESSION_SUMMARY.json").read_text(encoding="utf-8"))
    audit = json.loads((root / "IMPLEMENTATION_VALUE_AUDIT.json").read_text(encoding="utf-8"))
    if audit["local_energy_prices_assigned"] or audit["annual_dr_revenue_assigned"]:
        raise ValueError("Figure 6 requires the audited no-local-price, no-response-revenue boundary")

    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 215/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.070, top=0.855,
                        hspace=0.96, wspace=0.58)

    # a, the decision inequality carried by the implementation analysis.
    ax = axes[0, 0]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    title_artist = ax.text(0.0, 1.04, "(a) Value must cover resources", transform=ax.transAxes,
                           ha="left", va="bottom", fontsize=12, fontweight="bold")
    title_artist.set_gid("thermal-heterogeneity-panel-title")
    boxes = [
        (0.04, 0.58, 0.40, 0.24, PALE_GREEN, "Thermal service\nvalue"),
        (0.56, 0.58, 0.40, 0.24, PALE_BLUE, "Grid-response\nvalue"),
        (0.04, 0.16, 0.40, 0.24, PALE_ORANGE, "Implementation\nresources"),
        (0.56, 0.16, 0.40, 0.24, PALE_PURPLE, "Local energy\ncharge change"),
    ]
    for left, bottom, width, height, face, label in boxes:
        ax.add_patch(FancyBboxPatch((left, bottom), width, height,
                                    boxstyle="round,pad=0.015,rounding_size=0.025",
                                    facecolor=face, edgecolor="#777774", linewidth=0.8))
        ax.text(left+width/2, bottom+height/2, label, ha="center", va="center",
                fontsize=9.6, fontweight="bold")
    ax.text(0.50, 0.70, "+", ha="center", va="center", fontsize=14, fontweight="bold")
    ax.text(0.50, 0.28, "+", ha="center", va="center", fontsize=14, fontweight="bold")
    ax.text(0.50, 0.49, "≥", ha="center", va="center", fontsize=17, fontweight="bold")

    # b, the physical service-resource frontier before assigning monetary values.
    ax = axes[0, 1]
    final_title(ax, "b", "Service gains often\nraise energy")
    direct = boundaries.drop_duplicates(
        ["climate", "composition", "alternative_control",
         "target_service_gain_c", "delta_facility_purchased_kwh_year"]
    )
    for control, group in direct.groupby("alternative_control"):
        ax.scatter(group["target_service_gain_c"], group["delta_facility_purchased_kwh_year"]/1000,
                   s=27, color=CONTROL_COLORS[control], alpha=0.68,
                   edgecolor="white", linewidth=0.45, zorder=3)
    ax.axvline(0, color="#AAA9A5", linewidth=0.8, zorder=0)
    ax.set_xlim(-0.015, 0.29); ax.set_xticks([0.0, 0.1, 0.2])
    ax.set_xlabel("Reduction in population mismatch (°C)")
    ax.set_ylabel("Whole-building energy change (MWh)", labelpad=4)
    final_style(ax, zero=True, grid_axis="y")

    # c, command groups are a physical implementation resource.
    ax = axes[1, 0]
    final_title(ax, "c", "Command scope expands action")
    command_lookup = commands.set_index("command_scope")["independent_command_groups"].to_dict()
    permission = spatial[(spatial["axis"] == "permission") &
                         (spatial["reference"] == "building") &
                         spatial["control"].isin(["personal_center", "personal_range"])]
    for control in ("personal_center", "personal_range"):
        xs, medians, q1s, q3s = [0], [0.0], [0.0], [0.0]
        for alternative in ("floor", "native_zone"):
            values = -permission[(permission["control"] == control) &
                                 (permission["alternative"] == alternative)]["delta_mean_reference_mismatch_c"]
            q1, median, q3 = np.percentile(values, [25, 50, 75])
            xs.append(1 if alternative == "floor" else 2); medians.append(median); q1s.append(q1); q3s.append(q3)
        ax.plot(xs, medians, color=CONTROL_COLORS[control], marker="o", markersize=5,
                linewidth=1.7, zorder=4)
        ax.fill_between(xs, q1s, q3s, color=CONTROL_COLORS[control], alpha=0.16, zorder=2)
    ax.set_xticks([0, 1, 2], [f"Building\n{command_lookup['building']} group",
                               f"Floor\n{command_lookup['floor']} groups",
                               f"Zone\n{command_lookup['native_zone']} groups"])
    ax.set_xlabel("")
    ax.set_ylabel("Reduction in population mismatch (°C)", labelpad=1)
    final_style(ax, zero=True, grid_axis="y")

    # d, anonymous five-bin representation.
    ax = axes[1, 1]
    final_title(ax, "d", "Aggregate summaries\nretain most decisions")
    protected = compression["primary"]["protected five bins"]
    count_pairs = [
        (int(protected["selections"]), int(protected["scenarios"])),
        (int(protected["retained_full_qualified"]), int(protected["full_qualified"])),
        (int(protected["selected_plan_false_acceptances"]), int(protected["selections"])),
    ]
    labels = ["Cases with a\nselected plan", "Qualified candidate\nplans retained",
              "Selected plans\nmissing limits"]
    values = [numerator / denominator * 100 for numerator, denominator in count_pairs]
    colors = ["#4C82B8", "#56A17A", ACCENT_ORANGE]
    y = np.arange(3)[::-1]
    bars = ax.barh(y, values, color=colors, height=0.56, zorder=3)
    for bar, value, (numerator, denominator) in zip(bars, values, count_pairs):
        if value >= 25:
            x_text, alignment, text_color = value - 1.8, "right", "white"
        else:
            x_text, alignment, text_color = value + 1.5, "left", "black"
        text_artist = ax.text(x_text, bar.get_y()+bar.get_height()/2,
                              f"{numerator}/{denominator} ({value:.1f}%)",
                              ha=alignment, va="center", color=text_color)
        if text_color != "black":
            text_artist.set_gid("preserve-text-color")
    ax.set_yticks(y, labels, fontsize=8.8)
    ax.set_xlim(0, 105); ax.set_xticks([0, 25, 50, 75, 100]); ax.set_xlabel("Percentage (%)")
    final_style(ax, grid_axis="x")

    # e, explicit implementation-resource categories.
    ax = axes[2, 0]
    title_artist = ax.text(0.0, 1.04, "(e) Six resource categories", transform=ax.transAxes,
                           ha="left", va="bottom", fontsize=12, fontweight="bold")
    title_artist.set_gid("thermal-heterogeneity-panel-title")
    resource_labels = {
        "demand_acquisition": "Demand\nfeedback",
        "software_configuration": "Software setup",
        "ongoing_maintenance": "Maintenance",
        "staff_reconfiguration": "Occupant\nreassignment",
        "command_scope": "Command scope",
        "optional_local_hardware": "Local hardware",
    }
    catalog_order = list(resource_labels)
    catalog_index = catalog.set_index("resource_category")
    y = np.arange(len(catalog_order))[::-1]
    for yy, category in zip(y, catalog_order):
        row = catalog_index.loc[category]
        required = bool(row["required_in_every_deployment"])
        x = 0 if required else 1
        color = "#4C82B8" if required else ACCENT_PURPLE
        ax.scatter(x, yy, s=62, color=color, edgecolor="white", linewidth=0.6, zorder=3)
    ax.set_yticks(y, [resource_labels[item] for item in catalog_order], fontsize=8.2)
    ax.set_xticks([0, 1], ["Core resource", "Optional or\ncontext dependent"], fontsize=8.5)
    ax.set_xlim(-0.45, 1.45); ax.set_ylim(-0.7, len(catalog_order)-0.3)
    ax.grid(axis="x", color="#E6E6E3", linewidth=0.7, zorder=0)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#777777"); ax.spines["bottom"].set_color("#777777")
    ax.tick_params(length=0)

    # f, scenario-based resource sensitivity without local energy prices or response revenue.
    ax = axes[2, 1]
    final_title(ax, "f", "Annualized resource inputs")
    criteria = {
        "Setup labour\n40 h": (scenarios["setup_hours"].eq(40) & scenarios["annual_maintenance_hours"].eq(0) &
                                 scenarios["extra_responses_year"].eq(0) & scenarios["annual_software_usd"].eq(0) &
                                 scenarios["added_interface_capital_usd"].eq(0)),
        "Maintenance\n12 h year$^{-1}$": (scenarios["setup_hours"].eq(0) & scenarios["annual_maintenance_hours"].eq(12) &
                                        scenarios["extra_responses_year"].eq(0) & scenarios["annual_software_usd"].eq(0) &
                                        scenarios["added_interface_capital_usd"].eq(0)),
        "Feedback\n1,000 × 60 s": (scenarios["setup_hours"].eq(0) & scenarios["annual_maintenance_hours"].eq(0) &
                                           scenarios["extra_responses_year"].eq(1000) & scenarios["seconds_per_response"].eq(60) &
                                           scenarios["annual_software_usd"].eq(0) & scenarios["added_interface_capital_usd"].eq(0)),
        "Software\n500 USD yr⁻¹": (scenarios["setup_hours"].eq(0) & scenarios["annual_maintenance_hours"].eq(0) &
                                      scenarios["extra_responses_year"].eq(0) & scenarios["annual_software_usd"].eq(500) &
                                      scenarios["added_interface_capital_usd"].eq(0)),
        "Interface\n1,000 USD capital": (scenarios["setup_hours"].eq(0) & scenarios["annual_maintenance_hours"].eq(0) &
                                                    scenarios["extra_responses_year"].eq(0) & scenarios["annual_software_usd"].eq(0) &
                                                    scenarios["added_interface_capital_usd"].eq(1000)),
    }
    rows = []
    for label, mask in criteria.items():
        values = scenarios.loc[mask, "annual_resource_cost_usd"].drop_duplicates().to_numpy()
        rows.append((label, float(values.min()), float(np.median(values)), float(values.max())))
    y = np.arange(len(rows))[::-1]
    for yy, (label, low, median, high) in zip(y, rows):
        ax.hlines(yy, low, high, color="#9B9B97", linewidth=2.2, zorder=2)
        ax.scatter([low, high], [yy, yy], color="#9B9B97", s=22, zorder=3)
        ax.scatter(median, yy, color=ACCENT_PURPLE, s=42, edgecolor="white", linewidth=0.5, zorder=4)
        ax.text(high+18, yy, f"{low:.0f}–{high:.0f}", ha="left", va="center", fontsize=8.2)
    ax.set_yticks(y, [row[0] for row in rows], fontsize=8.1)
    ax.set_xlim(0, max(row[3] for row in rows)*1.35)
    ax.set_xticks([0, 200, 400, 600, 800])
    ax.set_xlabel("Annualized input (USD yr⁻¹)")
    final_style(ax, grid_axis="x")

    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      color=CONTROL_COLORS[c], label=CONTROL_LABELS[c])
               for c in ("personal_center", "personal_range")]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 0.982),
               frameon=False, facecolor="none", edgecolor="none", columnspacing=2.2)
    save_plot(fig, "Figure_06", size_mm=(180, 215))

draw_figure_06()


In [ ]:
# Supplementary Figure S1: archive flow and hierarchical demand variance.
def draw_figure_s01():
    variance = pd.read_csv(RESULTS / "support/science_audit/variance_recheck.csv")
    identity = pd.read_csv(RESULTS / "Building_identity/IDENTITY_VARIANCE_SENSITIVITY.csv")
    leave_one = pd.read_csv(RESULTS / "panel_data/Fig1h.csv")
    multinational = pd.read_csv(
        RESULTS / "legacy_analysis/human/human_profiles_minimized.csv.gz",
        usecols=["release_profile_id"],
    )
    china_hist = pd.read_csv(RESULTS / "CTCD_external/CTCD_REFERENCE_HISTOGRAM.csv")

    component_order = [
        "country", "city", "release_building_group_id", "release_unit_id", "within_unit"
    ]
    component_labels = {
        "country": "Country",
        "city": "City within country",
        "release_building_group_id": "Building within city",
        "release_unit_id": "People within building",
        "within_unit": "Repeated observation",
        "between_country": "Country",
        "between_city_within_country": "City within country",
        "between_building_within_city": "Building within city",
        "between_analysis_unit_within_building": "People within building",
        "within_analysis_unit": "Repeated observation",
    }
    identity_component = {
        "country": "country", "city": "city",
        "release_building_group_id": "building_id",
        "release_unit_id": "unit_group", "within_unit": "within_unit",
    }
    colors = dict(zip(component_order, ["#4C82B8", "#68A7A2", "#D7A85A", "#D47C4A", "#8066A8"]))
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 225/25.4))
    fig.subplots_adjust(left=0.205, right=0.98, bottom=0.065, top=0.88,
                        hspace=0.72, wspace=0.46)

    ax = axes[0, 0]
    counts = [len(multinational), int(china_hist["scope_records"].max())]
    bars = ax.bar([0, 1], counts, color=["#4C82B8", "#D47C4A"], width=0.62)
    for bar, value in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, value*1.02, f"{value:,}", ha="center", va="bottom")
    ax.set_xticks([0, 1], ["Multinational\narchive", "China\narchive"])
    ax.set_ylabel("Retained observations")
    ax.set_ylim(0, max(counts)*1.18)
    final_title(ax, "a", "Common archive flow")
    final_style(ax, grid_axis="y")

    ax = axes[0, 1]
    primary = variance[variance["cohort"].eq("all_canonical_records")].set_index("component")
    bottom = 0.0
    for key in component_order:
        value = float(primary.loc[key, "share"])*100
        ax.bar(0, value, bottom=bottom, color=colors[key], width=0.58)
        if value >= 6:
            ax.text(0, bottom+value/2, f"{value:.1f}%", ha="center", va="center", fontsize=9)
        bottom += value
    ax.set_xlim(-0.65, 0.65); ax.set_ylim(0, 100)
    ax.set_xticks([0], ["Primary\nspecification"]); ax.set_ylabel("Variance share (%)")
    final_title(ax, "b", "Primary variance shares")
    final_style(ax)

    ax = axes[1, 0]
    q = identity[(identity["subset"].eq("all_database_groups")) &
                 np.isclose(identity["griffiths_coefficient"], 0.50)]
    weighting_order = ["record_equal", "unit_equal", "building_equal", "source_equal"]
    for key in component_order:
        values = [float(q[(q.weighting.eq(weight)) & (q.component.eq(identity_component[key]))]["share"].iloc[0])*100
                  for weight in weighting_order]
        ax.plot(range(4), values, marker="o", lw=1.5, color=colors[key])
    ax.set_xticks(range(4), ["Record", "Person", "Building", "Source"])
    ax.set_ylabel("Variance share (%)")
    final_title(ax, "c", "Weighting sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[1, 1]
    q = identity[(identity["subset"].eq("all_database_groups")) &
                 (identity["weighting"].eq("unit_equal"))]
    coefficients = [0.33, 0.50, 0.67]
    for key in component_order:
        values = [float(q[np.isclose(q.griffiths_coefficient, coefficient) & q.component.eq(identity_component[key])]["share"].iloc[0])*100
                  for coefficient in coefficients]
        ax.plot(coefficients, values, marker="o", lw=1.5, color=colors[key])
    ax.set_xticks(coefficients); ax.set_xlabel("Griffiths coefficient (°C⁻¹)")
    ax.set_ylabel("Variance share (%)")
    final_title(ax, "d", "Reconstruction sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[2, 0]
    loo_order = [
        "between_country", "between_city_within_country", "between_building_within_city",
        "between_analysis_unit_within_building", "within_analysis_unit"
    ]
    q = leave_one[np.isclose(leave_one["griffiths"], 0.50) &
                  leave_one["weighting"].eq("analysis_unit_equal")].set_index("component")
    y = np.arange(len(loo_order))[::-1]
    med = np.array([float(q.loc[key, "loo_median_share"])*100 for key in loo_order])
    low = np.array([float(q.loc[key, "loo_min_share"])*100 for key in loo_order])
    high = np.array([float(q.loc[key, "loo_max_share"])*100 for key in loo_order])
    ax.errorbar(med, y, xerr=[med-low, high-med], fmt="o", color="#4C82B8",
                ecolor="#98B8CD", capsize=3)
    ax.set_yticks(y, ["Country", "City", "Building", "People", "Repeat"])
    ax.set_xlabel("Variance share (%)")
    final_title(ax, "e", "Leave-one-source range")
    final_style(ax, grid_axis="x")

    ax = axes[2, 1]
    subset_order = ["all_canonical_records", "known_subject_units_only"]
    for key in component_order:
        values = [float(variance[(variance.cohort.eq(subset)) &
                                 (variance.component.eq(key))]["share"].iloc[0])*100
                  for subset in subset_order]
        ax.plot([0, 1], values, marker="o", lw=1.5, color=colors[key])
    ax.set_xticks([0, 1], ["All records", "Identified\nsubset"])
    ax.set_xlim(-0.05, 1.12)
    ax.set_ylabel("Variance share (%)")
    final_title(ax, "f", "Identity sensitivity")
    final_style(ax, grid_axis="y")

    handles = [Patch(facecolor=colors[key], label=component_labels[key]) for key in component_order]
    fig.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.985),
               frameon=False, facecolor="none", edgecolor="none", columnspacing=1.2)
    save_plot(fig, "Figure_S01", size_mm=(180, 225))

draw_figure_s01()


In [ ]:
# Supplementary Figure S2: fixed reference temperatures and published guidance.
def draw_figure_s02():
    profiles = pd.read_csv(RESULTS / "legacy_analysis/human/human_profiles_minimized.csv.gz")
    policy = pd.read_csv(RESULTS / "legacy_analysis/policy/policy_range_estimates.csv")
    semantics = pd.read_csv(RESULTS / "legacy_analysis/policy/policy_semantics.csv")
    support = pd.read_csv(RESULTS / "legacy_analysis/policy/policy_support.csv")
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 230/25.4))
    fig.subplots_adjust(left=0.225, right=0.97, bottom=0.07, top=0.89,
                        hspace=0.72, wspace=0.48)
    coefficient_columns = [(0.33, "reference_g033_c", "#4C82B8"),
                           (0.50, "reference_g050_c", "#D7A85A"),
                           (0.67, "reference_g067_c", "#56A17A")]
    for panel, target, letter in zip(axes.flat[:3], [24, 25, 26], ["a", "b", "c"]):
        for coefficient, column, color in coefficient_columns:
            values = pd.to_numeric(profiles[column], errors="coerce").dropna() - target
            counts, edges = np.histogram(values, bins=np.linspace(-12, 12, 73), density=True)
            centres = (edges[:-1]+edges[1:])/2
            panel.plot(centres, counts, lw=1.5, color=color, label=f"{coefficient:.2f} °C⁻¹")
        panel.axvline(0, color="#8F8F8B", lw=0.8, ls="--")
        panel.set_xlim(-10, 10); panel.set_xlabel(f"Demand centre minus {target} °C (°C)")
        panel.set_ylabel("Density")
        final_title(panel, letter, f"Reference at {target} °C")
        final_style(panel)

    ax = axes[1, 1]
    range_rows = policy[policy["class"].isin(["below", "within", "above"])].copy()
    labels = (range_rows[["policy_id", "plot_label", "plot_order"]].drop_duplicates()
              .sort_values("plot_order"))
    labels = labels.head(9)
    compact_policy_labels = {
        "Indonesia Permenkes 48": "Indonesia rule",
        "Malaysia ICOP IAQ": "Malaysia code",
        "Australia model COP": "Australia code",
        "Canada CCOHS": "Canada",
        "India ECSBC 2024": "India ECBCS",
        "Singapore SS 553": "Singapore",
        "Sweden AFS 2023:12": "Sweden rule",
        "USA OSHA guidance": "USA OSHA",
    }
    y = np.arange(len(labels))[::-1]
    class_colors = {"below": "#4C82B8", "within": "#56A17A", "above": "#D47C4A"}
    left = np.zeros(len(labels))
    for class_name in ["below", "within", "above"]:
        values = []
        for policy_id in labels.policy_id:
            row = range_rows[(range_rows.policy_id.eq(policy_id)) &
                             (range_rows["class"].eq(class_name)) &
                             (range_rows.g_label.eq("g0.50"))]
            values.append(float(row.proportion.iloc[0])*100 if len(row) else 0)
        ax.barh(y, values, left=left, color=class_colors[class_name], height=0.62,
                label=class_name.title())
        left += np.asarray(values)
    ax.set_yticks(y, [compact_policy_labels.get(item, item) for item in labels.plot_label]); ax.set_xlim(0, 100)
    ax.set_xlabel("Observations (%)")
    final_title(ax, "d", "Published ranges")
    final_style(ax, grid_axis="x")

    ax = axes[2, 0]
    ranges = semantics[semantics.lower_c.notna() & semantics.upper_c.notna()].sort_values("plot_order")
    y = np.arange(len(ranges))[::-1]
    for yy, row in zip(y, ranges.itertuples()):
        color = "#4C82B8" if row.layer == "formal_range" else "#8066A8"
        ax.hlines(yy, row.lower_c, row.upper_c, color=color, lw=5)
        ax.scatter([row.lower_c, row.upper_c], [yy, yy], s=24, color=color, zorder=3)
    ax.set_yticks(y, [compact_policy_labels.get(item, item) for item in ranges.plot_label]); ax.set_xlabel("Published temperature (°C)")
    final_title(ax, "e", "Range metric and scope")
    final_style(ax, grid_axis="x")

    ax = axes[2, 1]
    targets = semantics[semantics.target_c.notna()].merge(
        support[["policy_id", "records"]], on="policy_id", how="left"
    ).sort_values("plot_order")
    y = np.arange(len(targets))[::-1]
    sizes = 30 + 90*np.sqrt(targets.records.fillna(0)/max(float(targets.records.max()), 1.0))
    ax.scatter(targets.target_c, y, s=sizes, color="#D47C4A", alpha=0.82,
               edgecolor="white", linewidth=0.6)
    compact_target_labels = {
        "China summer rule": "China summer",
        "China winter rule": "China winter",
        "Thailand EGAT campaign": "Thailand rule",
        "Philippines AO 15": "Philippines",
        "UK HSE minimum": "UK minimum",
    }
    ax.set_yticks(y, [compact_target_labels.get(item, item) for item in targets.plot_label])
    ax.set_xlim(14, 29); ax.set_xticks([16, 20, 24, 28])
    ax.set_xlabel("Published one-sided target (°C)")
    final_title(ax, "f", "One-sided targets")
    final_style(ax, grid_axis="x")

    coeff_handles = [Line2D([0], [0], color=color, lw=1.8, label=f"{coefficient:.2f} °C⁻¹")
                     for coefficient, _, color in coefficient_columns]
    class_handles = [Patch(facecolor=class_colors[key], label=key.title())
                     for key in ["below", "within", "above"]]
    fig.legend(handles=coeff_handles+class_handles, loc="upper center", ncol=6,
               bbox_to_anchor=(0.5, 0.985), frameon=False, facecolor="none", edgecolor="none")
    save_plot(fig, "Figure_S02", size_mm=(180, 230))

draw_figure_s02()


In [ ]:
# Supplementary Figure S3: geographic demand patterns and transfer diagnostics.
def draw_figure_s03():
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    from matplotlib.path import Path as MplPath
    from thermal_heterogeneity.figures import draw_world_land, load_world_land

    support = pd.read_csv(RESULTS / "legacy_analysis/curves/city_season_demand_support.csv")
    holdout = pd.read_csv(RESULTS / "Supporting_diagnostics/spatial_interpolation_city_holdout.csv")
    masks = pd.read_csv(RESULTS / "Supporting_diagnostics/spatial_interpolation_support_masks.csv")
    support = support[support.supported.astype(bool)].copy()
    support = support.dropna(subset=["map_longitude_0p5deg", "map_latitude_0p5deg", "g050_p50_c"])
    cmap = plt.get_cmap("RdYlBu_r")
    norm = Normalize(vmin=float(support.g050_p50_c.quantile(0.03)),
                     vmax=float(support.g050_p50_c.quantile(0.97)))
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 185/25.4))
    fig.subplots_adjust(left=0.10, right=0.93, bottom=0.09, top=0.90,
                        hspace=0.52, wspace=0.42)

    def base_map(ax, letter, title):
        draw_world_land(ax, facecolor="#F3F3F0", edgecolor="#555555", linewidth=0.32)
        ax.set_xlabel(""); ax.set_ylabel("")
        ax.tick_params(labelsize=9)
        final_title(ax, letter, title)

    for ax, season, letter in zip(axes[0], ["warm", "cold"], ["a", "b"]):
        q = support[support.season_group.eq(season)]
        base_map(ax, letter, f"Observed {season}-season centres")
        sizes = 18 + 95*np.sqrt(q.analysis_unit_count / max(float(q.analysis_unit_count.max()), 1))
        ax.scatter(q.map_longitude_0p5deg, q.map_latitude_0p5deg, c=q.g050_p50_c,
                   cmap=cmap, norm=norm, s=sizes, edgecolor="white", linewidth=0.45, zorder=3)

    land_paths = [MplPath(poly) for poly in load_world_land()]
    lon_grid, lat_grid = np.meshgrid(np.arange(-177.5, 180, 5.0), np.arange(-57.5, 87.6, 5.0))
    grid = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
    land_mask = np.zeros(len(grid), dtype=bool)
    for polygon in land_paths:
        land_mask |= polygon.contains_points(grid)

    def haversine(points, observations):
        p = np.radians(points); o = np.radians(observations)
        dlat = p[:, None, 1] - o[None, :, 1]
        dlon = p[:, None, 0] - o[None, :, 0]
        a = np.sin(dlat/2)**2 + np.cos(p[:, None, 1])*np.cos(o[None, :, 1])*np.sin(dlon/2)**2
        return 6371.0 * 2*np.arctan2(np.sqrt(a), np.sqrt(np.maximum(1-a, 0)))

    for ax, season, letter in zip(axes[1], ["warm", "cold"], ["c", "d"]):
        q = support[support.season_group.eq(season)]
        observed = q[["map_longitude_0p5deg", "map_latitude_0p5deg"]].to_numpy(float)
        distances = haversine(grid, observed)
        nearest = np.argsort(distances, axis=1)[:, :4]
        selected = np.take_along_axis(distances, nearest, axis=1)
        weights = 1/np.maximum(selected, 1.0)**2
        values = q.g050_p50_c.to_numpy(float)[nearest]
        prediction = np.sum(values*weights, axis=1)/np.sum(weights, axis=1)
        retained = land_mask & (distances.min(axis=1) <= 1000)
        base_map(ax, letter, f"{season.title()}-season interpolation")
        ax.scatter(grid[retained, 0], grid[retained, 1], c=prediction[retained], cmap=cmap,
                   norm=norm, s=18, marker="s", linewidth=0, alpha=0.88, zorder=1)
        ax.scatter(observed[:, 0], observed[:, 1], c=q.g050_p50_c, cmap=cmap, norm=norm,
                   s=22, edgecolor="black", linewidth=0.35, zorder=3)

    ax = axes[2, 0]
    season_colors = {"warm": "#D47C4A", "cold": "#4C82B8"}
    for season, q in holdout.groupby("season"):
        ax.scatter(q.observed_c, q.heldout_prediction_c, s=31, color=season_colors[season],
                   alpha=0.75, edgecolor="white", linewidth=0.4, label=season.title())
    limits = [min(holdout.observed_c.min(), holdout.heldout_prediction_c.min())-0.4,
              max(holdout.observed_c.max(), holdout.heldout_prediction_c.max())+0.4]
    ax.plot(limits, limits, ls="--", color="#777777", lw=0.9)
    ax.set_xlim(limits); ax.set_xticks([16, 20, 24, 28, 32]); ax.set_ylim(limits)
    ax.set_xlabel("Observed city median (°C)"); ax.set_ylabel("Held-out prediction (°C)")
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(ax, "e", "Whole-city holdout")
    final_style(ax, grid_axis="both")

    ax = axes[2, 1]
    for season, q in holdout.groupby("season"):
        ax.scatter(q.nearest_training_city_km, q.absolute_error_c, s=30,
                   color=season_colors[season], alpha=0.72, edgecolor="white", linewidth=0.4)
    for radius, style in zip([500, 1000, 1500], [":", "--", "-."]):
        ax.axvline(radius, color="#8C8C89", lw=0.8, ls=style)
    ax.set_xlim(0, max(holdout.nearest_training_city_km.max()*1.05, 1000)); ax.set_xticks([0, 2000, 4000, 6000, 8000])
    ax.set_xlabel("Nearest training city (km)"); ax.set_ylabel("Absolute error (°C)")
    ax.set_ylim(-0.1, max(1.0, holdout.absolute_error_c.max() * 1.28))
    ax.text(0.98, 0.96, "Support radii: 500, 1,000, 1,500 km", transform=ax.transAxes,
            ha="right", va="top", fontsize=9)
    final_title(ax, "f", "Transfer distance and error")
    final_style(ax, grid_axis="y")

    colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=axes[:2, :].ravel().tolist(),
                            location="right", fraction=0.025, pad=0.025)
    colorbar.ax.set_gid("thermal-heterogeneity-global-legend-axis")
    colorbar.set_label("Demand centre (°C)")
    save_plot(fig, "Figure_S03", size_mm=(180, 185))

draw_figure_s03()


In [ ]:
# Supplementary Figure S4: reconstruction and demographic sensitivity.
def draw_figure_s04():
    audit = RESULTS / "support/spatial_audit"
    metrics = pd.read_csv(audit / "metrics_summary.csv")
    predictions = pd.read_csv(audit / "oof_predictions.csv")
    paired = pd.read_csv(audit / "paired_cluster_bootstrap_vs_mean.csv")
    gender = pd.read_csv(RESULTS / "support/science_audit/demographic_city_season_gender_summary.csv")
    age = pd.read_csv(RESULTS / "support/science_audit/demographic_city_season_age_summary.csv")
    coverage = pd.read_csv(RESULTS / "support/science_audit/demographic_coverage.csv")
    model_order = ["global_training_mean", "idw2_4nn", "gpr_spherical_xyz"]
    model_labels = ["Training mean", "Four-neighbour IDW", "Spherical GP"]
    model_colors = ["#73777A", "#4C82B8", "#56A17A"]
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 225/25.4))
    fig.subplots_adjust(left=0.16, right=0.98, bottom=0.07, top=0.90,
                        hspace=0.70, wspace=0.43)

    primary = metrics[(metrics.target.eq("g050_p50_c")) & (metrics.season_scope.eq("all"))]
    for ax, field, ylabel, letter, title in [
        (axes[0, 0], "mae_c", "Mean absolute error (°C)", "a", "Held-out absolute error"),
        (axes[0, 1], "rmse_c", "Root-mean-square error (°C)", "b", "Held-out squared error"),
    ]:
        values = [float(primary[primary.model.eq(model)][field].iloc[0]) for model in model_order]
        bars = ax.bar(range(3), values, color=model_colors, width=0.64)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x()+bar.get_width()/2, value+0.04, f"{value:.2f}", ha="center", va="bottom")
        ax.set_xticks(range(3), model_labels, rotation=18, ha="right")
        ax.set_ylabel(ylabel); ax.set_ylim(0, max(values)*1.24)
        final_title(ax, letter, title); final_style(ax, grid_axis="y")

    ax = axes[1, 0]
    q = predictions[(predictions.target.eq("g050_p50_c")) &
                    (predictions.model.eq("gpr_spherical_xyz"))]
    covered = q.covered_95.astype("string").str.lower().isin(["true", "1"])
    ax.errorbar(q.observed_c, q.prediction_c,
                yerr=[q.prediction_c-q.lower_95_c, q.upper_95_c-q.prediction_c],
                fmt="none", ecolor="#B7B7B3", alpha=0.62, lw=0.7)
    ax.scatter(q.observed_c[covered], q.prediction_c[covered], s=28, color="#4C82B8",
               edgecolor="white", linewidth=0.4, label="Covered")
    ax.scatter(q.observed_c[~covered], q.prediction_c[~covered], s=34, color="#D47C4A",
               edgecolor="white", linewidth=0.4, label="Missed")
    limits = [min(q.observed_c.min(), q.prediction_c.min())-0.5,
              max(q.observed_c.max(), q.prediction_c.max())+0.5]
    ax.plot(limits, limits, ls="--", lw=0.8, color="#777777")
    ax.set_xlim(limits); ax.set_ylim(limits)
    ax.set_xlabel("Observed city median (°C)"); ax.set_ylabel("Predicted city median (°C)")
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(ax, "c", "Prediction intervals")
    final_style(ax, grid_axis="both")

    ax = axes[1, 1]
    target_coefficients = {"g033_p50_c": 0.33, "g050_p50_c": 0.50, "g067_p50_c": 0.67}
    q = paired[(paired.season_scope.eq("all")) & paired.model.isin(model_order[1:])]
    for model, color, marker in [("idw2_4nn", "#4C82B8", "o"),
                                 ("gpr_spherical_xyz", "#56A17A", "s")]:
        r = q[q.model.eq(model)].copy()
        r["coefficient"] = r.target.map(target_coefficients)
        r = r.sort_values("coefficient")
        ax.errorbar(r.coefficient, r.delta_mae_c_model_minus_mean,
                    yerr=[r.delta_mae_c_model_minus_mean-r.delta_mae_c_ci_low,
                          r.delta_mae_c_ci_high-r.delta_mae_c_model_minus_mean],
                    color=color, marker=marker, capsize=3, lw=1.3,
                    label="Four-neighbour" if model == "idw2_4nn" else "Spherical GP")
    ax.axhline(0, color="#888884", lw=0.8, ls="--")
    ax.set_xticks([0.33, 0.50, 0.67]); ax.set_xlabel("Griffiths coefficient (°C⁻¹)")
    ax.set_ylabel("Change in held-out error (°C)")
    low, high = ax.get_ylim()
    ax.set_ylim(low, max(1.0, high))
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper center", ncol=1)
    final_title(ax, "d", "Reconstruction sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[2, 0]
    contrasts = []
    for coefficient in [0.33, 0.50, 0.67]:
        column = f"reference_center_g{int(round(coefficient*100)):03d}_median_c"
        pivot = gender.pivot_table(index=["country", "city", "season"], columns="gender", values=column)
        contrasts.append((pivot.get("female")-pivot.get("male")).dropna().to_numpy())
    box = ax.boxplot(contrasts, positions=range(3), widths=0.55, patch_artist=True, showfliers=False)
    for patch, color in zip(box["boxes"], ["#4C82B8", "#D7A85A", "#56A17A"]):
        patch.set_facecolor(color); patch.set_alpha(0.72)
    ax.set_xticks(range(3), ["0.33", "0.50", "0.67"])
    ax.set_xlabel("Griffiths coefficient (°C⁻¹)"); ax.set_ylabel("Female minus male centre (°C)")
    final_title(ax, "e", "Paired gender contrasts")
    final_style(ax, zero=True, grid_axis="y")

    ax = axes[2, 1]
    age_order = [item for item in ["under_20", "20_29", "30_39", "40_49", "50_59", "60_plus"]
                 if item in set(age.age_band)]
    values = [age[age.age_band.eq(group)].reference_center_g050_median_c.dropna().to_numpy()
              for group in age_order]
    box = ax.boxplot(values, positions=range(len(age_order)), widths=0.55,
                     patch_artist=True, showfliers=False)
    for patch in box["boxes"]:
        patch.set_facecolor("#8066A8"); patch.set_alpha(0.60)
    labels = [item.replace("under_", "<").replace("_plus", "+").replace("_", "–")
              for item in age_order]
    ax.set_xticks(range(len(labels)), labels, rotation=25, ha="right")
    ax.set_ylabel("City–season median centre (°C)")
    age_cov = float(coverage[coverage.field.eq("age")].record_coverage.iloc[0])*100
    gender_cov = float(coverage[coverage.field.eq("gender")].record_coverage.iloc[0])*100
    low, high = ax.get_ylim()
    span = high-low
    ax.set_ylim(low, high+0.32*span)
    ax.text(0.97, 0.96, f"Record coverage\nAge {age_cov:.0f}% · gender {gender_cov:.0f}%",
            transform=ax.transAxes, ha="right", va="top", linespacing=1.12)
    final_title(ax, "f", "Age and field coverage")
    final_style(ax, grid_axis="y")

    save_plot(fig, "Figure_S04", size_mm=(180, 225))

draw_figure_s04()


In [ ]:
# Supplementary Figure S5: shared-room thermal requests.
def draw_figure_s05():
    windows = pd.read_csv(RESULTS / "legacy_analysis/human/direct_vote_windows_sanitized.csv.gz")
    windows = windows[windows.eligible.astype(bool)].copy()
    categories = pd.read_csv(RESULTS / "Figure_sources/figure_01_shared_room_source.csv")
    dataset_order = [item for item in ["NUS_HAAS", "ENTH"] if item in set(windows.dataset_code)]
    dataset_labels = {"NUS_HAAS": "Singapore", "ENTH": "Longitudinal"}
    request_colors = {"Cooler": "#4C82B8", "No change": "#D7A85A", "Warmer": "#D47C4A"}
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 220/25.4))
    fig.subplots_adjust(left=0.18, right=0.98, bottom=0.07, top=0.89,
                        hspace=0.70, wspace=0.43)

    for ax, dataset, letter in zip(axes[0], dataset_order, ["a", "b"]):
        q = windows[windows.dataset_code.eq(dataset)].copy()
        q["warmer_share"] = q.prefer_warmer_count/q.vote_count
        q["no_change_share"] = q.no_change_count/q.vote_count
        q["cooler_share"] = q.prefer_cooler_count/q.vote_count
        q = q.sort_values(["warmer_share", "no_change_share", "cooler_share"]).reset_index(drop=True)
        x = np.arange(len(q)); bottom = np.zeros(len(q))
        for label, column in [("Cooler", "cooler_share"), ("No change", "no_change_share"),
                              ("Warmer", "warmer_share")]:
            values = q[column].to_numpy()*100
            ax.bar(x, values, bottom=bottom, width=1.0, color=request_colors[label], linewidth=0)
            bottom += values
        ax.set_xlim(-0.5, len(q)-0.5); ax.set_ylim(0, 100)
        ax.set_xlabel("Eligible 30-minute room windows", labelpad=7); ax.set_ylabel("Preference responses (%)")
        ax.set_xticks([])
        final_title(ax, letter, f"{dataset_labels[dataset]} room windows")
        final_style(ax)

    ax = axes[1, 0]
    category_order = ["Same", "Mixed", "Opposed"]
    width = 0.34; x = np.arange(3)
    for index, dataset in enumerate(dataset_order):
        q = categories[categories.dataset.eq(dataset)].set_index("category")
        values = [int(q.loc[item, "window_count"]) if item in q.index else 0 for item in category_order]
        ax.bar(x+(index-0.5)*width, values, width=width,
               color="#4C82B8" if index == 0 else "#D47C4A",
               label=dataset_labels[dataset])
    ax.set_xticks(x, category_order); ax.set_ylabel("Eligible windows")
    ax.set_ylim(0, max(1, categories.window_count.max()) * 1.45)
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper center", ncol=2)
    final_title(ax, "c", "Coexisting request categories")
    final_style(ax, grid_axis="y")

    ax = axes[1, 1]
    for dataset, marker, color in [(dataset_order[0], "o", "#4C82B8"),
                                   (dataset_order[1], "s", "#D47C4A")]:
        q = windows[windows.dataset_code.eq(dataset)]
        ax.scatter(q.full_burden, q.compressed_burden, s=30, marker=marker,
                   color=color, alpha=0.72, edgecolor="white", linewidth=0.45,
                   label=dataset_labels[dataset])
    ax.plot([0, 1], [0, 1], ls="--", color="#777777", lw=0.8)
    ax.set_xlim(-0.03, 1.03); ax.set_xticks([0, 0.25, 0.50, 0.75, 1.00]); ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel("Full request burden"); ax.set_ylabel("Burden after averaging")
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(ax, "d", "Full versus averaged burden")
    final_style(ax, grid_axis="both")

    ax = axes[2, 0]
    values = [windows[windows.dataset_code.eq(dataset)].jensen_gap.to_numpy() for dataset in dataset_order]
    violin = ax.violinplot(values, positions=range(len(values)), showmeans=False,
                           showmedians=True, showextrema=False)
    for body, color in zip(violin["bodies"], ["#4C82B8", "#D47C4A"]):
        body.set_facecolor(color); body.set_alpha(0.65); body.set_edgecolor("none")
    violin["cmedians"].set_color("black")
    ax.set_xticks(range(len(values)), [dataset_labels[item] for item in dataset_order])
    ax.set_ylabel("Full minus averaged burden")
    final_title(ax, "e", "Information removed per window")
    final_style(ax, zero=True, grid_axis="y")

    ax = axes[2, 1]
    metric_names = ["Different requests", "Opposing requests", "Exact cancellation"]
    width = 0.34; x = np.arange(3)
    for index, dataset in enumerate(dataset_order):
        q = windows[windows.dataset_code.eq(dataset)]
        values = [q.disagreement_present.mean()*100,
                  q.simultaneous_warmer_cooler.mean()*100,
                  ((q.compressed_burden < 1e-12) & (q.full_burden > 0)).mean()*100]
        ax.bar(x+(index-0.5)*width, values, width=width,
               color="#4C82B8" if index == 0 else "#D47C4A")
    ax.set_xticks(x, ["Different", "Opposed", "Cancelled"])
    ax.set_ylabel("Eligible windows (%)"); ax.set_ylim(0, 100)
    ax.text(0.98, 0.97, "Independent room-window evidence", transform=ax.transAxes,
            ha="right", va="top", fontsize=9)
    final_title(ax, "f", "Replication across studies")
    final_style(ax, grid_axis="y")

    request_handles = [Patch(facecolor=request_colors[item], label=item)
                       for item in ["Cooler", "No change", "Warmer"]]
    fig.legend(handles=request_handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.985),
               frameon=False, facecolor="none", edgecolor="none")
    save_plot(fig, "Figure_S05", size_mm=(180, 220))

draw_figure_s05()


In [ ]:
# Supplementary Figure S6: observations to six fixed demand compositions.
def draw_figure_s06():
    profiles = pd.read_csv(ROOT / "output/global_climate_matrix_fixed_v2_40iter/GLOBAL_PROFILES.csv")
    compositions = (profiles.sort_values(["context", "composition_id", "profile_id"])
                    .drop_duplicates(["composition_id", "profile_id"]).copy())
    compositions["range_width_c"] = compositions.controller_upper_c-compositions.controller_lower_c
    archive = pd.read_csv(RESULTS / "legacy_analysis/human/human_profiles_minimized.csv.gz",
                          usecols=["release_profile_id"])
    china_hist = pd.read_csv(RESULTS / "CTCD_external/CTCD_REFERENCE_HISTOGRAM.csv")
    climate_order = [item for item in CONTEXT_ORDER if item in set(profiles.context)]
    cohort_order = list(dict.fromkeys(compositions.cohort_label.tolist()))
    source_colors = {"Multinational archive": "#4C82B8", "China archive": "#D47C4A"}
    cohort_colors = {label: ("#4C82B8" if "Multinational" in label else "#D47C4A")
                     for label in cohort_order}
    cohort_linestyles = {"A": "-", "B": "--", "C": ":"}
    fig, axes = plt.subplots(3, 2, figsize=(180/25.4, 225/25.4))
    fig.subplots_adjust(left=0.18, right=0.98, bottom=0.07, top=0.89,
                        hspace=0.72, wspace=0.48)

    ax = axes[0, 0]
    retained = [len(archive), int(china_hist.scope_records.max())]
    selected = [int((compositions.demand_source.eq(source)).sum())
                for source in ["Multinational archive", "China archive"]]
    x = np.arange(2); width = 0.34
    ax.bar(x-width/2, retained, width=width, color=["#4C82B8", "#D47C4A"], alpha=0.35,
           label="Retained observations")
    ax.bar(x+width/2, selected, width=width, color=["#4C82B8", "#D47C4A"],
           label="Assigned profiles")
    ax.set_xticks(x, ["Multinational", "China"]); ax.set_ylabel("Observations or profiles")
    ax.set_yscale("log")
    ax.set_ylim(600, 70000)
    ax.set_yticks([1_000, 10_000]); ax.set_yticklabels(["1,000", "10,000"])
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper center", ncol=1)
    final_title(ax, "a", "From observations to compositions")
    final_style(ax, grid_axis="y")

    ax = axes[0, 1]
    bins = np.linspace(compositions.centre_c.min(), compositions.centre_c.max(), 42)
    for label in cohort_order:
        q = compositions[compositions.cohort_label.eq(label)].centre_c
        density, edges = np.histogram(q, bins=bins, density=True)
        ax.plot((edges[:-1]+edges[1:])/2, density, color=cohort_colors[label], lw=1.35,
                ls=cohort_linestyles[label.rsplit(" ", 1)[-1]], alpha=0.90, label=label)
    ax.set_xlim(12.5, 33.0); ax.set_xticks([15, 20, 25, 30])
    ax.set_xlabel("Demand centre (°C)"); ax.set_ylabel("Density")
    final_title(ax, "b", "Reference centres")
    final_style(ax)

    ax = axes[1, 0]
    values = [compositions[compositions.cohort_label.eq(label)].range_width_c.to_numpy()
              for label in cohort_order]
    box = ax.boxplot(values, positions=range(len(values)), widths=0.55,
                     patch_artist=True, showfliers=False)
    for patch, label in zip(box["boxes"], cohort_order):
        patch.set_facecolor(cohort_colors[label]); patch.set_alpha(0.62)
    cohort_letters = [label.rsplit(" ", 1)[-1] for label in cohort_order]
    ax.set_xticks(range(len(cohort_order)), cohort_letters, rotation=0, ha="center")
    ax.text(1, -0.17, "Multinational", transform=ax.get_xaxis_transform(),
            ha="center", va="top")
    ax.text(4, -0.17, "China", transform=ax.get_xaxis_transform(),
            ha="center", va="top")
    ax.set_ylabel("Personal range width (°C)")
    final_title(ax, "c", "Personal range widths")
    final_style(ax, grid_axis="y")

    ax = axes[1, 1]
    example_label = cohort_order[0]
    example = compositions[compositions.cohort_label.eq(example_label)].sort_values("centre_c").reset_index(drop=True)
    y = np.arange(len(example))
    ax.hlines(y, example.controller_lower_c, example.controller_upper_c,
              color="#A9C6D8", lw=0.8, alpha=0.75)
    ax.scatter(example.centre_c, y, s=9, color="#4C82B8", edgecolor="none")
    ax.set_xlim(15.5, 30.5); ax.set_xticks([16, 20, 24, 28])
    ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Profiles ordered by centre")
    ax.set_yticks([0, len(example)//2, len(example)-1], ["1", str(len(example)//2+1), str(len(example))])
    final_title(ax, "d", "Example centres and ranges")
    final_style(ax, grid_axis="x")

    ax = axes[2, 0]
    slots = profiles[(profiles.context.eq(climate_order[0])) &
                     (profiles.cohort_label.eq(example_label))].groupby("zone_id").profile_id.nunique().sort_values()
    def zone_floor(label):
        parts = label.split("_")
        token = parts[1]
        return {"bottom": "Bottom floor", "bot": "Bottom floor",
                "mid": "Middle floor", "top": "Top floor"}[token]
    floor_order = ["Bottom floor", "Middle floor", "Top floor"]
    floor_slots = slots.groupby([zone_floor(label) for label in slots.index]).sum().reindex(floor_order)
    y = np.arange(len(floor_slots))
    ax.barh(y, floor_slots.values, color="#56A17A")
    ax.set_yticks(y, [f"{label} total" for label in floor_slots.index])
    ax.set_xlabel("Assigned profile slots")
    final_title(ax, "e", "Profile slots by floor")
    final_style(ax, grid_axis="x")

    ax = axes[2, 1]
    availability = (profiles.groupby(["cohort_label", "context"]).profile_id.nunique()
                    .unstack(fill_value=0).reindex(index=cohort_order, columns=climate_order))
    image = ax.imshow((availability.to_numpy() > 0).astype(int), aspect="auto",
                      cmap=matplotlib.colors.ListedColormap(["#F1F1EE", "#8066A8"]),
                      vmin=0, vmax=1)
    ax.set_xticks(range(len(climate_order)), [CONTEXT_LABELS[item] for item in climate_order],
                  rotation=35, ha="right")
    ax.set_yticks(range(len(cohort_order)), [label.replace("Multinational ", "Multinational ") for label in cohort_order])
    ax.set_xticks(np.arange(-0.5, len(climate_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(cohort_order), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)
    final_title(ax, "f", "Cohort–climate coverage")
    for spine in ax.spines.values():
        spine.set_color("#777777")

    handles = [Line2D([0], [0], color=cohort_colors[label], lw=1.8,
                      ls=cohort_linestyles[label.rsplit(" ", 1)[-1]], label=label)
               for label in cohort_order]
    fig.legend(handles=handles, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.985),
               frameon=False, facecolor="none", edgecolor="none", columnspacing=1.2)
    save_plot(fig, "Figure_S06", size_mm=(180, 225))

draw_figure_s06()


In [ ]:
# Supplementary Figure S7: Personal range validation and fixed-width endpoints.
def draw_figure_s07():
    validation = RESULTS / "global_extension/personal_range_validation"
    summary = pd.read_csv(validation / "VIRTUAL_WIDTH_HELDOUT_SUMMARY.csv")
    development = pd.read_csv(validation / "OBSERVED_RANGE_DEVELOPMENT.csv")
    external = pd.read_csv(validation / "ENTH_EXTERNAL_SUMMARY.csv")
    fixed = pd.read_csv(RESULTS / "global_extension/fixed_range_endpoint_sensitivity/FIXED_RANGE_ENDPOINT_DIRECT_COMPARISONS.csv")
    model_order = ["virtual_width", "fixed_1C", "fixed_1.5C", "fixed_2C", "fixed_2.5C", "fixed_3C"]
    model_labels = ["Published", "±1.0°C", "±1.5°C", "±2.0°C", "±2.5°C", "±3.0°C"]
    fig, axes = plt.subplots(4, 2, figsize=(180/25.4, 255/25.4))
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.06, top=0.91,
                        hspace=0.82, wspace=0.43)

    ax = axes[0, 0]
    records = [int(summary[summary.outcome.eq("acceptability")].records.iloc[0]),
               int(summary[summary.outcome.eq("preference_binary")].records.iloc[0]),
               int(external.records.max())]
    people = [int(summary[summary.outcome.eq("acceptability")].people.iloc[0]),
              int(summary[summary.outcome.eq("preference_binary")].people.iloc[0]),
              int(external.people.max())]
    x = np.arange(3); width = 0.35
    ax.bar(x-width/2, records, width=width, color="#4C82B8", label="Repeated observations")
    ax.bar(x+width/2, people, width=width, color="#D47C4A", label="People")
    ax.set_xticks(x, ["Acceptability", "Preference", "External study"])
    ax.set_ylabel("Count")
    ax.set_ylim(0, max(records) * 1.55)
    ax.legend(frameon=False, facecolor="none", edgecolor="none", loc="upper right")
    final_title(ax, "a", "Chronological and external evaluation")
    final_style(ax, grid_axis="y")

    for ax, outcome, letter, title in [
        (axes[0, 1], "acceptability", "b", "Acceptability classification"),
        (axes[1, 0], "preference_binary", "c", "No-change preference classification"),
    ]:
        q = summary[summary.outcome.eq(outcome)].set_index("range_model").reindex(model_order)
        colors = ["#8066A8"] + ["#A9BFD0"]*(len(model_order)-1)
        bars = ax.bar(range(len(q)), q.balanced_accuracy*100, color=colors, width=0.66)
        for bar, value in zip(bars, q.balanced_accuracy*100):
            ax.text(bar.get_x()+bar.get_width()/2, value+1.0, f"{value:.1f}", ha="center", va="bottom", fontsize=8.5)
        ax.set_xticks(range(len(q)), model_labels, rotation=18, ha="right")
        ax.set_ylabel("Balanced accuracy (%)"); ax.set_ylim(0, 78)
        final_title(ax, letter, title); final_style(ax, grid_axis="y")

    ax = axes[1, 1]
    values = []
    labels = []
    colors = []
    for outcome, short in [("acceptability", "Accept."), ("preference_binary", "No change")]:
        for model, model_label, color in [("virtual_width", "Published", "#8066A8"),
                                          ("fixed_2C", "Fixed ±2°C", "#56A17A")]:
            values.append(float(summary[(summary.outcome.eq(outcome)) &
                                        (summary.range_model.eq(model))].roc_auc.iloc[0])*100)
            compact = "Accept." if outcome == "acceptability" else "No chg."
            labels.append(f"{compact}\n{'Pub.' if model == 'virtual_width' else '±2°C'}")
            colors.append(color)
    ax.bar(range(4), values, color=colors, width=0.65)
    ax.set_xticks(range(4), labels)
    ax.set_ylabel("Receiver-operating area (%)"); ax.set_ylim(0, 82)
    final_title(ax, "d", "Discrimination across outcomes")
    final_style(ax, grid_axis="y")

    for ax, field, letter, title, ylabel in [
        (axes[2, 0], "delta_fixed_interval_coverage", "e", "Fixed-width coverage change", "Coverage change (pp)"),
        (axes[2, 1], "delta_fixed_interval_mean_excess_c", "f", "Fixed-width interval excess", "Interval-excess change (°C)"),
    ]:
        positions = []; values = []; colors = []; labels = []
        position = 0
        for half_width in [1.5, 2.0]:
            for control in ["personal_center", "personal_range"]:
                q = fixed[np.isclose(fixed.fixed_half_width_c, half_width) &
                          fixed.alternative_control.eq(control)][field].dropna().to_numpy()
                positions.append(position); values.append(q); colors.append(CONTROL_COLORS[control])
                labels.append(f"±{half_width:.1f}°C\n{CONTROL_LABELS[control].replace('Personal ', '')}")
                position += 1
            position += 0.5
        box = ax.boxplot(values, positions=positions, widths=0.58, patch_artist=True, showfliers=False)
        for patch, color in zip(box["boxes"], colors):
            patch.set_facecolor(color); patch.set_alpha(0.68)
        ax.set_xticks(positions, labels); ax.set_ylabel(ylabel)
        final_title(ax, letter, title); final_style(ax, zero=True, grid_axis="y")

    ax = axes[3, 0]
    rows = []
    for half_width in [1.5, 2.0]:
        q = fixed[np.isclose(fixed.fixed_half_width_c, half_width)]
        rows.append((f"±{half_width:.1f}°C coverage", int((q.delta_fixed_interval_coverage > 0).sum()), len(q)))
        rows.append((f"±{half_width:.1f}°C excess", int((q.delta_fixed_interval_mean_excess_c < 0).sum()), len(q)))
    y = np.arange(len(rows))[::-1]
    bars = ax.barh(y, [100*good/total for _, good, total in rows], color=["#4C82B8", "#56A17A"]*2)
    ax.set_yticks(y, [label for label, _, _ in rows]); ax.set_xlim(0, 100)
    ax.set_xlabel("Comparisons improving service (%)")
    final_title(ax, "g", "Service direction is retained")
    final_style(ax, grid_axis="x")

    ax = axes[3, 1]
    labels = []; increase = []; decrease = []
    for half_width in [1.5, 2.0]:
        for control in ["personal_center", "personal_range"]:
            q = fixed[np.isclose(fixed.fixed_half_width_c, half_width) &
                      fixed.alternative_control.eq(control)]
            labels.append(f"±{half_width:.1f}°C\n{CONTROL_LABELS[control].replace('Personal ', '')}")
            increase.append(int((q.delta_facility_purchased_pct > 0).sum()))
            decrease.append(int((q.delta_facility_purchased_pct < 0).sum()))
    x = np.arange(len(labels))
    ax.bar(x, increase, color="#D47C4A", label="Higher energy")
    ax.bar(x, decrease, bottom=increase, color="#4C82B8", label="Lower energy")
    ax.set_xticks(x, labels); ax.set_ylabel("Paired comparisons")
    ax.set_ylim(0, max(np.asarray(increase)+np.asarray(decrease))*1.28)
    handles = [Patch(facecolor="#D47C4A", label="Higher"),
               Patch(facecolor="#4C82B8", label="Lower")]
    ax.legend(handles=handles, frameon=False, facecolor="none", edgecolor="none",
              loc="upper center", ncol=2)
    final_title(ax, "h", "Energy direction by width")
    final_style(ax, grid_axis="y")

    save_plot(fig, "Figure_S07", size_mm=(180, 255))

draw_figure_s07()


In [ ]:
# Supplementary Figure S8: fixed building and nine-climate experiment.
def draw_figure_s08():
    from collections import Counter
    from thermal_heterogeneity.figures import draw_world_land

    weather = pd.read_csv(ROOT / "data/metadata/global_fixed_reference_weather_sources.csv")
    provenance = json.loads((ROOT / "data/metadata/global_fixed_reference_inputs.provenance.json").read_text(encoding="utf-8"))
    parameters = json.loads((ROOT / "data/metadata/experimental_parameter_provenance.json").read_text(encoding="utf-8"))
    metrics = pd.read_csv(RESULTS / "global_extension/global_final/GLOBAL_RUN_METRICS.csv")
    prototype = provenance["prototype"]
    fig, axes = plt.subplots(4, 2, figsize=(180/25.4, 255/25.4))
    fig.subplots_adjust(left=0.16, right=0.98, bottom=0.055, top=0.91,
                        hspace=0.80, wspace=0.43)

    ax = axes[0, 0]
    draw_world_land(ax, facecolor="#F1F1EE", edgecolor="#555555", linewidth=0.32)
    climate_colors = {zone: plt.get_cmap("tab10")(index) for index, zone in enumerate(sorted(weather.ashrae_climate_zone.unique()))}
    for city_number, row in enumerate(weather.itertuples(), start=1):
        ax.scatter(row.longitude, row.latitude, marker="*", s=95,
                   color=climate_colors[row.ashrae_climate_zone], edgecolor="black", linewidth=0.5, zorder=3)
        ax.annotate(str(city_number), (row.longitude, row.latitude), xytext=(5, 5),
                    textcoords="offset points", ha="left", va="bottom", fontsize=7.5,
                    fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    final_title(ax, "a", "Nine weather contexts")

    ax = axes[0, 1]
    zones = sorted(weather.ashrae_climate_zone.unique())
    counts = weather.ashrae_climate_zone.value_counts().reindex(zones)
    bars = ax.bar(range(len(zones)), counts, color=[climate_colors[zone] for zone in zones])
    ax.set_xticks(range(len(zones)), zones); ax.set_ylabel("Weather contexts")
    ax.set_ylim(0, max(counts)+0.7)
    for bar, value in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, value+0.05, str(int(value)), ha="center", va="bottom")
    final_title(ax, "b", "Eight ASHRAE climate zones")
    final_style(ax, grid_axis="y")

    ax = axes[1, 0]
    ax.set_xlim(0, 5); ax.set_ylim(0, 3.35); ax.axis("off")
    for floor in range(3):
        for zone in range(5):
            ax.add_patch(Rectangle((zone+0.05, floor+0.08), 0.86, 0.76,
                                   facecolor=[PALE_BLUE, PALE_GREEN, PALE_ORANGE][floor],
                                   edgecolor="#777777", linewidth=0.7))
            ax.text(zone+0.48, floor+0.46, str(zone+1), ha="center", va="center", fontsize=8.5)
        ax.text(-0.05, floor+0.46, f"Floor {floor+1}", ha="right", va="center")
    ax.text(2.5, 3.25, f"{prototype['floor_area_m2']:,.0f} m²; 15 occupied zones",
            ha="center", va="top")
    final_title(ax, "c", "Fixed reference-building layout")

    ax = axes[1, 1]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    hvac_boxes = [(0.04, 0.65, 0.26, 0.20, PALE_BLUE, "3 VAV\nloops"),
                  (0.37, 0.65, 0.26, 0.20, PALE_ORANGE, "Gas\nheating"),
                  (0.70, 0.65, 0.26, 0.20, PALE_GREEN, "DX cooling"),
                  (0.22, 0.22, 0.56, 0.22, PALE_PURPLE, "15 electric-reheat\nterminals")]
    for left, bottom, width, height, color, label in hvac_boxes:
        ax.add_patch(FancyBboxPatch((left, bottom), width, height,
                                    boxstyle="round,pad=0.01,rounding_size=0.02",
                                    facecolor=color, edgecolor="#777777", linewidth=0.8))
        ax.text(left+width/2, bottom+height/2, label, ha="center", va="center", fontsize=9)
    for x in [0.17, 0.50, 0.83]:
        ax.annotate("", xy=(0.50, 0.45), xytext=(x, 0.65),
                    arrowprops=dict(arrowstyle="->", lw=0.8, color="#777777"))
    final_title(ax, "d", "Heating and cooling topology")

    ax = axes[2, 0]
    categories = ["Geometry", "Envelope", "Internal loads", "HVAC", "Weather", "Equipment size"]
    status = ["Fixed", "Fixed", "Fixed", "Fixed", "Climate-specific", "Autosized"]
    y = np.arange(len(categories))[::-1]
    status_colors = {"Fixed": "#4C82B8", "Climate-specific": "#D47C4A", "Autosized": "#8066A8"}
    for yy, category, state in zip(y, categories, status):
        ax.scatter([0], [yy], s=70, color=status_colors[state], edgecolor="white", linewidth=0.6)
        ax.text(0.10, yy, state, va="center", ha="left")
    ax.set_yticks(y, categories); ax.set_xlim(-0.08, 0.80); ax.set_xticks([])
    final_title(ax, "e", "Fixed and climate-sized inputs")
    final_style(ax)

    ax = axes[2, 1]
    dimensions = [metrics.climate.nunique(), metrics.composition.nunique(), metrics.control.nunique(), metrics.offset_c.nunique()]
    labels = ["Climates", "Compositions", "Controls", "Offsets"]
    bars = ax.bar(range(4), dimensions, color=["#4C82B8", "#D47C4A", "#56A17A", "#8066A8"])
    for bar, value in zip(bars, dimensions):
        ax.text(bar.get_x()+bar.get_width()/2, value+0.2, str(int(value)), ha="center", va="bottom")
    ax.set_xticks(range(4), ["Climate\ncontexts", "Demand\ngroups", "Controls", "Offsets"], fontsize=9)
    ax.set_ylabel("Design levels")
    ax.text(0.98, 0.95, f"{len(metrics):,} completed candidates", transform=ax.transAxes,
            ha="right", va="top", fontweight="bold")
    final_title(ax, "f", "Annual experiment matrix")
    final_style(ax, grid_axis="y")

    ax = axes[3, 0]
    offsets = sorted(metrics.offset_c.unique())
    for offset in offsets:
        centre = 22.5+offset
        ax.hlines(offset, centre-1.5, centre+1.5, color="#4C82B8", lw=4)
        ax.scatter([centre-1.5, centre+1.5], [offset, offset], color="#D47C4A", s=22, zorder=3)
    ax.set_yticks(offsets, [f"{value:+.1f}°C" for value in offsets])
    ax.set_xlabel("Translated thermostat target (°C)"); ax.set_ylabel("Candidate offset")
    final_title(ax, "g", "Prototype deadband and offsets")
    final_style(ax, grid_axis="x")

    ax = axes[3, 1]
    classes = Counter(item["classification"] for item in parameters["parameters"].values())
    readable = {
        "prototype_inherited": "Prototype",
        "literature_supported_scenario": "Literature",
        "study_defined_sensitivity": "Study design",
        "data_derived_sensitivity": "Data-derived",
        "pre_control_selection_rule": "Selection rule",
        "accounting_horizon": "Accounting",
    }
    items = sorted(classes.items(), key=lambda pair: (-pair[1], pair[0]))
    y = np.arange(len(items))[::-1]
    ax.barh(y, [value for _, value in items], color="#8066A8")
    ax.set_yticks(y, [readable.get(key, key.replace("_", " ").title()) for key, _ in items])
    ax.set_xlim(0, 8.8); ax.set_xticks([0, 2, 4, 6, 8])
    ax.set_xlabel("Documented parameters")
    final_title(ax, "h", "Parameter provenance classes")
    final_style(ax, grid_axis="x")

    save_plot(fig, "Figure_S08", size_mm=(180, 255))

draw_figure_s08()


In [ ]:
# Supplementary Figure S9: plan qualification and information sensitivity.
def draw_figure_s09():
    base = RESULTS / "global_extension"
    information = pd.read_csv(base / "global_final/GLOBAL_INFORMATION_DIAGNOSTIC.csv")
    tail = pd.read_csv(base / "tail_fraction_sensitivity/TAIL_FRACTION_INFORMATION_DIAGNOSTIC.csv")
    anchors = pd.read_csv(base / "reference_anchor_sensitivity/REFERENCE_ANCHOR_HEADLINE.csv")
    compression = pd.read_csv(base / "information_compression_final/GLOBAL_INFORMATION_COMPRESSION_DECISIONS.csv")
    fig, axes = plt.subplots(4, 2, figsize=(180/25.4, 255/25.4))
    fig.subplots_adjust(left=0.16, right=0.98, bottom=0.06, top=0.88,
                        hspace=0.82, wspace=0.46)

    ax = axes[0, 0]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    boxes = [(0.03, 0.58, 0.27, 0.24, PALE_BLUE, "Physical\nplans"),
             (0.37, 0.58, 0.27, 0.24, PALE_ORANGE, "Zone-mean\nreview"),
             (0.70, 0.58, 0.27, 0.24, PALE_GREEN, "All-profile\nreview"),
             (0.37, 0.15, 0.27, 0.22, PALE_PURPLE, "Eligibility +\nselected plan")]
    for left, bottom, width, height, color, label in boxes:
        ax.add_patch(FancyBboxPatch((left, bottom), width, height,
                                    boxstyle="round,pad=0.01,rounding_size=0.02",
                                    facecolor=color, edgecolor="#777777", linewidth=0.8))
        ax.text(left+width/2, bottom+height/2, label, ha="center", va="center", fontsize=9)
    for start, end in [((0.30, 0.70), (0.37, 0.70)), ((0.64, 0.70), (0.70, 0.70)),
                       ((0.50, 0.58), (0.50, 0.37)), ((0.83, 0.58), (0.60, 0.37))]:
        ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=0.8, color="#777777"))
    final_title(ax, "a", "Identical plans, different information")

    primary = information[np.isclose(information.service_fraction, 1.0)].copy()
    ax = axes[0, 1]
    distribution = int(primary.eligibility_disagreement.sum())
    agreement = int(len(primary)-distribution)
    ax.bar([0, 1], [agreement, distribution], color=["#56A17A", "#D47C4A"], width=0.62)
    for x, value in enumerate([agreement, distribution]):
        ax.text(x, value+1, str(value), ha="center", va="bottom")
    ax.set_xticks([0, 1], ["Same eligibility", "Distribution\nchanges eligibility"])
    ax.set_ylabel("Climate–composition cells"); ax.set_ylim(0, 60)
    final_title(ax, "b", "Primary decision classes")
    final_style(ax, grid_axis="y")

    ax = axes[1, 0]
    failures = (primary.groupby("climate").zone_mean_selected_plan_fails_full_review.sum()
                .reindex(CONTEXT_ORDER, fill_value=0))
    bars = ax.bar(range(len(failures)), failures, color="#D47C4A")
    ax.set_xticks(range(len(failures)), [CONTEXT_LABELS[item] for item in failures.index],
                  rotation=35, ha="right")
    ax.set_ylabel("Selections missing limits", labelpad=14)
    final_title(ax, "c", "Missed-limit selections by climate")
    final_style(ax, grid_axis="y")

    ax = axes[1, 1]
    tail_primary = tail[np.isclose(tail.service_fraction, 1.0)]
    rows = []
    for fraction in sorted(tail_primary.tail_fraction.unique()):
        q = tail_primary[np.isclose(tail_primary.tail_fraction, fraction)]
        rows.append((fraction, int(q.eligibility_disagreement.sum()),
                     int(q.zone_mean_selected_plan_fails_full_review.sum())))
    x = np.arange(len(rows)); width = 0.34
    ax.bar(x-width/2, [row[1] for row in rows], width=width, color="#4C82B8", label="Eligibility differs")
    ax.bar(x+width/2, [row[2] for row in rows], width=width, color="#D47C4A", label="Selected plan misses limits")
    ax.set_xticks(x, [f"{row[0]*100:.0f}%" for row in rows]); ax.set_xlabel("Fixed least-served share")
    ax.set_ylabel("Climate–composition cells")
    final_title(ax, "d", "Tail-share sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[2, 0]
    x = np.arange(len(anchors)); width = 0.34
    ax.bar(x-width/2, anchors.eligibility_disagreement_at_fraction_1, width=width,
           color="#4C82B8", label="Eligibility differs")
    ax.bar(x+width/2, anchors.zone_mean_selected_plan_fails_full_review_at_fraction_1,
           width=width, color="#D47C4A", label="Selected plan misses limits")
    ax.set_xticks(x, [f"{value:.0f}°C" for value in anchors.reference_target_c])
    ax.set_xlabel("External reference"); ax.set_ylabel("Climate–composition cells")
    final_title(ax, "e", "Reference-anchor sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[2, 1]
    domain_specs = [("20–27°C", "target_domain_final", "declared_20_27_c"),
                    ("20–26°C", "target_domain_20_26", "central_20_26_c"),
                    ("21–27°C", "target_domain_21_27", "central_21_27_c"),
                    ("21–26°C", "target_domain_21_26", "central_21_26_c")]
    domain_rows = []
    for label, folder, scope in domain_specs:
        q = pd.read_csv(base / folder / "TARGET_DOMAIN_INFORMATION_DIAGNOSTIC.csv")
        q = q[np.isclose(q.service_fraction, 1.0) & q.domain_scope.eq(scope)]
        domain_rows.append((label, int(q.eligibility_disagreement.sum()),
                            int(q.zone_mean_selected_plan_fails_full_review.sum())))
    x = np.arange(len(domain_rows)); width = 0.34
    ax.bar(x-width/2, [row[1] for row in domain_rows], width=width, color="#4C82B8")
    ax.bar(x+width/2, [row[2] for row in domain_rows], width=width, color="#D47C4A")
    ax.set_xticks(x, [row[0].replace("°C", "\n°C") for row in domain_rows]); ax.set_xlabel("Retained target domain")
    ax.set_ylabel("Climate–composition cells")
    final_title(ax, "f", "Target-domain sensitivity")
    final_style(ax, grid_axis="y")

    ax = axes[3, 0]
    target_summary = pd.read_csv(base / "target_domain_final/TARGET_DOMAIN_SUMMARY.csv")
    rates = (target_summary.groupby("control").base_optimizer_touches_declared_boundary.mean()
             .reindex(CONTROL_ORDER) * 100)
    bars = ax.bar(range(3), rates, color=[CONTROL_COLORS[item] for item in CONTROL_ORDER])
    for bar, value in zip(bars, rates):
        ax.text(bar.get_x()+bar.get_width()/2, value+1.4, f"{value:.0f}%", ha="center", va="bottom")
    ax.set_xticks(range(3), ["Uniform\naverage", "Personal\ncentre", "Personal\nrange"])
    ax.set_ylabel("Offset-zero optimizers at boundary (%)"); ax.set_ylim(0, 100)
    final_title(ax, "g", "Target-boundary contact")
    final_style(ax, grid_axis="y")

    ax = axes[3, 1]
    q = compression[np.isclose(compression.service_fraction, 1.0)]
    representation_order = [item for item in ["zone mean", "five bins", "full profiles"]
                            if item in set(q.representation)]
    exact = [100*q[q.representation.eq(item)].exact_selection_agreement.mean() for item in representation_order]
    false_accept = [int(q[q.representation.eq(item)].candidate_false_accept_count.sum())
                    for item in representation_order]
    x = np.arange(len(representation_order))
    bars = ax.bar(x, exact, color=["#73777A", "#4C82B8", "#56A17A"][:len(x)])
    for bar, value in zip(bars, exact):
        ax.text(bar.get_x()+bar.get_width()/2, value-8, f"{value:.0f}%",
                ha="center", va="top", color="white", fontweight="bold")
    ax.set_xticks(x, [item.title() for item in representation_order])
    ax.set_ylabel("Exact selected-plan agreement (%)"); ax.set_ylim(0, 118)
    final_title(ax, "h", "Information compression")
    final_style(ax, grid_axis="y")

    decision_handles = [Patch(facecolor="#4C82B8", label="Eligibility differs"),
                        Patch(facecolor="#D47C4A", label="Selected plan misses limits")]
    fig.legend(handles=decision_handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 0.982),
               frameon=False, facecolor="none", edgecolor="none")
    save_plot(fig, "Figure_S09", size_mm=(180, 255))

draw_figure_s09()


In [ ]:
# Supplementary Figure S10: annual modeled thermal service.
def draw_figure_s10():
    direct = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_DIRECT_COMPARISONS.csv")
    direct["archive"] = np.where(direct.composition.str.contains("China", case=False),
                                 "China archive", "Multinational archive")
    control_order = ["personal_center", "personal_range"]
    endpoint_rows = [
        ("delta_mean_reference_mismatch_c", -1.0, "Reduction in population mismatch (°C)", "Population mismatch"),
        ("delta_fixed_tail_reference_mismatch_c", -1.0, "Reduction for least-served group (°C)", "Least-served group"),
        ("delta_modeled_interval_person_hour_coverage", 100.0, "Coverage change (percentage points)", "Personal range coverage"),
        ("delta_modeled_interval_mean_excess_c", -1.0, "Reduction in interval excess (°C)", "Personal range excess"),
    ]
    fig, axes = plt.subplots(4, 2, figsize=(180/25.4, 260/25.4), sharex=True)
    fig.subplots_adjust(left=0.17, right=0.985, bottom=0.075, top=0.91,
                        hspace=0.84, wspace=0.38)
    climate_order = [item for item in CONTEXT_ORDER if item in set(direct.climate)]
    marker_by_archive = {"Multinational archive": "o", "China archive": "s"}

    for row_index, (field, multiplier, ylabel, endpoint_title) in enumerate(endpoint_rows):
        for column_index, control in enumerate(control_order):
            ax = axes[row_index, column_index]
            q = direct[direct.alternative_control.eq(control)].copy()
            for climate_index, climate in enumerate(climate_order):
                group = q[q.climate.eq(climate)]
                for archive, archive_group in group.groupby("archive"):
                    values = archive_group[field].to_numpy(float)*multiplier
                    jitter = deterministic_jitter(len(values), width=0.12)
                    ax.scatter(np.full(len(values), climate_index)+jitter, values,
                               s=24, marker=marker_by_archive[archive],
                               color=CONTROL_COLORS[control], alpha=0.72,
                               edgecolor="white", linewidth=0.4, zorder=3)
                if len(group):
                    median = float(np.median(group[field].to_numpy(float)*multiplier))
                    ax.plot([climate_index-0.18, climate_index+0.18], [median, median],
                            color="black", lw=1.2, zorder=4)
            ax.axhline(0, color="#999995", lw=0.8, ls="--")
            ax.set_ylabel(ylabel if column_index == 0 else "")
            if row_index == 3:
                ax.set_xticks(range(len(climate_order)), [CONTEXT_LABELS[item] for item in climate_order],
                              rotation=35, ha="right")
            else:
                ax.set_xticks(range(len(climate_order)), [])
            letter = chr(97+row_index*2+column_index)
            short_control = "Centre" if control == "personal_center" else "Range"
            short_endpoint = {"Population mismatch": "Population mismatch",
                              "Least-served group": "Least-served group",
                              "Personal range coverage": "Range coverage",
                              "Personal range excess": "Range excess"}[endpoint_title]
            final_title(ax, letter, f"{short_control}: {short_endpoint}")
            final_style(ax, grid_axis="y")

    handles = [Line2D([0], [0], marker=marker_by_archive[archive], linestyle="", color="#5A5A58",
                      label=archive, markersize=6) for archive in marker_by_archive]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 0.985),
               frameon=False, facecolor="none", edgecolor="none")
    save_plot(fig, "Figure_S10", size_mm=(180, 260))

draw_figure_s10()


In [ ]:
# Figure_S11: direct whole-building energy and operational-carbon outcomes.
def draw_s11():
    d = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_DIRECT_COMPARISONS.csv")
    d = d[d["status"].eq("matched")].copy()
    controls = ["personal_center", "personal_range"]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.10, top=0.88, hspace=0.60, wspace=0.48)
    rng = np.random.default_rng(2026102711)

    for ax, column, scale, ylabel, letter, title in [
        (axes[0, 0], "delta_facility_purchased_pct", 1.0, "Whole-building energy change (%)", "a", "Energy change"),
        (axes[0, 1], "delta_operational_co2_country_kg", 1/1000, "Operational carbon change (t CO₂e)", "b", "Carbon change"),
    ]:
        for index, control in enumerate(controls):
            values = d.loc[d.alternative_control.eq(control), column].to_numpy(float) * scale
            ax.scatter(index + rng.normal(0, 0.055, len(values)), values, s=22,
                       color=CONTROL_COLORS[control], alpha=0.65, edgecolor="white", linewidth=0.35)
            add_median_iqr(ax, index, values, CONTROL_COLORS[control])
        ax.set_xticks([0, 1], ["Personal\ncentre", "Personal\nrange"])
        ax.set_ylabel(ylabel, labelpad=3)
        final_title(ax, letter, title)
        final_style(ax, zero=True, grid_axis="y")

    medians = d.groupby("alternative_control")[["delta_facility_electric_kwh", "delta_facility_gas_hhv_kwh"]].median().reindex(controls) / 1000
    x = np.arange(2); width = 0.34
    axes[1, 0].bar(x-width/2, medians.delta_facility_electric_kwh, width, color="#4C82B8", label="Electricity")
    axes[1, 0].bar(x+width/2, medians.delta_facility_gas_hhv_kwh, width, color="#D47C4A", label="Gas input")
    axes[1, 0].set_xticks(x, ["Personal\ncentre", "Personal\nrange"])
    axes[1, 0].set_ylabel("Median annual change (MWh)")
    axes[1, 0].set_ylim(0, float(medians.delta_facility_electric_kwh.max())*1.45)
    axes[1, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[1, 0], "c", "Energy carriers")
    final_style(axes[1, 0], zero=True, grid_axis="y")

    for control in controls:
        q = d[d.alternative_control.eq(control)]
        axes[1, 1].scatter(q.delta_facility_purchased_kwh/1000,
                           q.delta_operational_co2_country_kg/1000,
                           s=23, color=CONTROL_COLORS[control], alpha=0.68,
                           edgecolor="white", linewidth=0.35, label=CONTROL_LABELS[control])
    axes[1, 1].set_xlim(-10, 48); axes[1, 1].set_xticks([0, 20, 40])
    axes[1, 1].set_xlabel("Whole-building energy change (MWh)")
    axes[1, 1].set_ylabel("Operational carbon change (t CO₂e)")
    final_title(axes[1, 1], "d", "Energy–carbon relation")
    final_style(axes[1, 1], zero=True, grid_axis="y")

    caption = ("Supplementary Figure S11 | Direct annual resource outcomes across nine climates and six demand groups. "
               "a,b, Whole-building purchased-energy and country-specific operational-carbon changes for Personal centre and Personal range relative to Uniform average. Points are matched comparisons; black marks show medians and interquartile ranges. "
               "c, Median changes in facility electricity and gas input. d, Paired whole-building energy and operational-carbon changes.")
    (RESULTS / "captions/Figure_S11.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S11", size_mm=(180, 136))

draw_s11()


In [ ]:
# Figure_S12: matched-service and strict-service sensitivity.
def draw_s12():
    matched = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_MATCHED_SERVICE.csv")
    strict = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_STRICT_SERVICE_FEASIBILITY.csv")
    controls = ["uniform_average", "personal_center", "personal_range"]
    alternatives = ["personal_center", "personal_range"]
    tolerances = [0.01, 0.025, 0.05, 0.075, 0.10]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.11, top=0.90, hspace=0.61, wspace=0.48)

    for control in alternatives:
        counts = [int(matched[f"within_tolerance_{str(t).replace('.', 'p').replace('0p1', '0p100')}_c"].sum()) if False else 0 for t in tolerances]
        columns = ["within_tolerance_0p010_c", "within_tolerance_0p025_c", "within_tolerance_0p050_c", "within_tolerance_0p075_c", "within_tolerance_0p100_c"]
        counts = [int(matched.loc[matched.alternative_control.eq(control), column].sum()) for column in columns]
        axes[0, 0].plot(np.arange(5), counts, "o-", color=CONTROL_COLORS[control], linewidth=1.8,
                        label=CONTROL_LABELS[control])
    axes[0, 0].set_xticks(np.arange(5), ["0.01", "0.025", "0.05", "0.075", "0.10"])
    axes[0, 0].set_xlabel("Maximum service gap (°C)")
    axes[0, 0].set_ylim(0, 52)
    axes[0, 0].set_ylabel("Matched comparisons")
    axes[0, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[0, 0], "a", "Comparable service expands with tolerance")
    final_style(axes[0, 0], grid_axis="y")

    q = matched[matched.within_tolerance_0p100_c].copy()
    x = np.arange(2)
    lower = [int(((q.alternative_control.eq(c)) & q.energy_direction.eq("decrease")).sum()) for c in alternatives]
    higher = [int(((q.alternative_control.eq(c)) & q.energy_direction.eq("increase")).sum()) for c in alternatives]
    axes[0, 1].bar(x, lower, color="#4C82B8", label="Energy lower")
    axes[0, 1].bar(x, higher, bottom=lower, color="#D47C4A", label="Energy higher")
    axes[0, 1].set_xticks(x, ["Personal\ncentre", "Personal\nrange"])
    axes[0, 1].set_ylabel("Matched comparisons")
    axes[0, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    axes[0, 1].set_ylim(0, 65); axes[0, 1].set_yticks([0, 20, 40, 60])
    final_title(axes[0, 1], "b", "Energy direction at 0.10°C")
    final_style(axes[0, 1], grid_axis="y")

    for control in alternatives:
        z = q[q.alternative_control.eq(control)]
        axes[1, 0].scatter(z.service_linf_gap_c, z.delta_energy_pct, s=24,
                           color=CONTROL_COLORS[control], alpha=0.70,
                           edgecolor="white", linewidth=0.35, label=CONTROL_LABELS[control])
    axes[1, 0].set_xlabel("Maximum service gap (°C)")
    axes[1, 0].set_ylabel("Whole-building energy change (%)")
    final_title(axes[1, 0], "c", "Energy within the matched-service set")
    final_style(axes[1, 0], zero=True, grid_axis="y")

    selected = strict.assign(selected=strict.selection_status.eq("selected"))
    fractions = [1.0, 0.75, 0.5]
    width = 0.24; xx = np.arange(3)
    for idx, control in enumerate(controls):
        values = [int(selected[selected.service_fraction.eq(f) & selected.control.eq(control)].selected.sum()) for f in fractions]
        axes[1, 1].bar(xx+(idx-1)*width, values, width, color=CONTROL_COLORS[control], label=CONTROL_LABELS[control])
    axes[1, 1].set_xticks(xx, ["1.00", "0.75", "0.50"])
    axes[1, 1].set_xlabel("Service-limit fraction")
    axes[1, 1].set_ylim(0, 32); axes[1, 1].set_yticks([0, 5, 10, 15, 20])
    axes[1, 1].set_ylabel("Conditions with a selected plan")
    axes[1, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper right")
    final_title(axes[1, 1], "d", "Feasibility under tighter limits")
    final_style(axes[1, 1], grid_axis="y")

    caption = ("Supplementary Figure S12 | Service-comparable and strict-service sensitivity. a, Number of demand-guided comparisons within each maximum service-gap tolerance. "
               "b, Energy direction among the 67 comparisons within 0.10 °C. c, Energy change across that near-matched-service set. d, Conditions retaining a selected operating plan as the service limit is tightened.")
    (RESULTS / "captions/Figure_S12.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S12", size_mm=(180, 136))

draw_s12()


In [ ]:
# Figure_S13: seasonal redistribution and incomplete cancellation.
def draw_s13():
    d = pd.read_csv(RESULTS / "global_extension/final_story/FINAL_SEASONAL_REALLOCATION.csv")
    seasons = [("winter", "delta_winter_facility_purchased_kwh"),
               ("spring", "delta_spring_facility_purchased_kwh"),
               ("summer", "delta_summer_facility_purchased_kwh"),
               ("autumn", "delta_autumn_facility_purchased_kwh")]
    colors = ["#76A7C5", "#63A788", "#D47C4A", "#C7A15B"]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.10, top=0.90, hspace=0.58, wspace=0.46)

    for index, (label, column) in enumerate(seasons):
        values = d[column].dropna().to_numpy()/1000
        parts = axes[0, 0].violinplot(values, positions=[index], widths=0.74, showextrema=False, showmedians=True)
        for body in parts["bodies"]:
            body.set_facecolor(colors[index]); body.set_alpha(0.74)
        parts["cmedians"].set_color("black")
    axes[0, 0].set_xticks(range(4), [s[0].title() for s in seasons])
    axes[0, 0].set_ylabel("Seasonal energy change (MWh)")
    final_title(axes[0, 0], "a", "Energy moves across seasons")
    final_style(axes[0, 0], zero=True, grid_axis="y")

    positive = []; negative = []
    for _, column in seasons:
        values = d[column].dropna()
        positive.append((values > 0).mean()*100); negative.append((values < 0).mean()*100)
    axes[0, 1].bar(range(4), negative, color="#4C82B8", label="Energy lower")
    axes[0, 1].bar(range(4), positive, bottom=negative, color="#D47C4A", label="Energy higher")
    axes[0, 1].set_xticks(range(4), [s[0].title() for s in seasons])
    axes[0, 1].set_ylim(0, 160); axes[0, 1].set_yticks([0, 25, 50, 75, 100]); axes[0, 1].set_ylabel("Comparisons (%)")
    axes[0, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left", ncol=1, bbox_to_anchor=(0.0, 0.995))
    final_title(axes[0, 1], "b", "Direction changes with season")
    final_style(axes[0, 1], grid_axis="y")

    for fraction, marker, label in [(0.5, "o", "Half contrast"), (1.0, "s", "Full contrast")]:
        q = d[d.seasonal_contrast_fraction.eq(fraction)]
        axes[1, 0].scatter(-q.delta_annual_mean_reference_mismatch_c,
                           q.delta_annual_facility_purchased_pct, s=23,
                           marker=marker, color=ACCENT_PURPLE if fraction == 1 else "#76A7C5",
                           alpha=0.68, edgecolor="white", linewidth=0.35, label=label)
    axes[1, 0].set_xlabel("Reduction in annual population mismatch (°C)")
    axes[1, 0].set_ylabel("Whole-building energy change (%)")
    lower, upper = axes[1, 0].get_ylim(); axes[1, 0].set_ylim(lower, upper+7)
    axes[1, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[1, 0], "c", "Annual energy gap persists")
    final_style(axes[1, 0], zero=True, grid_axis="y")

    for index, (fraction, color) in enumerate([(0.5, "#76A7C5"), (1.0, ACCENT_PURPLE)]):
        values = d.loc[d.seasonal_contrast_fraction.eq(fraction), "seasonal_energy_cancellation_fraction"]*100
        axes[1, 1].scatter(index + deterministic_jitter(len(values), 0.12), values,
                           s=20, color=color, alpha=0.55, edgecolor="white", linewidth=0.3)
        add_median_iqr(axes[1, 1], index, values, color)
    axes[1, 1].set_xticks([0, 1], ["Half\ncontrast", "Full\ncontrast"])
    axes[1, 1].set_ylabel("Seasonal cancellation (%)")
    axes[1, 1].set_ylim(-3, 105)
    final_title(axes[1, 1], "d", "Cancellation remains limited")
    final_style(axes[1, 1], grid_axis="y")

    caption = ("Supplementary Figure S13 | Seasonal redistribution across nine climates. a, Winter, spring, summer and autumn changes in whole-building purchased energy. "
               "b, Shares of comparisons with lower or higher seasonal energy. c, Annual service and energy changes under half and full seasonal demand contrast. d, Cross-season cancellation of absolute energy changes.")
    (RESULTS / "captions/Figure_S13.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S13", size_mm=(180, 136))

draw_s13()


In [ ]:
# Figure_S14: occupant placement effects under fixed command permission.
def draw_s14():
    d = pd.read_csv(RESULTS / "global_extension/spatial_final/GLOBAL_PLACEMENT_COMPARISONS.csv")
    d = d[d.status.eq("matched") & d.control.isin(["personal_center", "personal_range"])].copy()
    metrics = [("delta_mean_reference_mismatch_c", -1, "Population mismatch reduction (°C)", "Population service"),
               ("delta_fixed_tail_reference_mismatch_c", -1, "Least-served mismatch reduction (°C)", "Least-served group"),
               ("delta_modeled_interval_person_hour_coverage", 100, "Coverage gain (points)", "Range coverage"),
               ("delta_facility_purchased_pct", 1, "Whole-building energy change (%)", "Energy consequence")]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 145/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.12, top=0.82, hspace=0.62, wspace=0.48)
    rng = np.random.default_rng(2026102714)
    alternatives = ["demand_ordered", "demand_ordered_reversed", "demand_ordered_shift5"]
    labels = ["Grouped", "Reversed", "Shifted"]
    for axis, (column, scale, ylabel, title), letter in zip(axes.flat, metrics, "abcd"):
        for alt_index, alternative in enumerate(alternatives):
            for control_index, control in enumerate(["personal_center", "personal_range"]):
                values = d[d.alternative.eq(alternative) & d.control.eq(control)][column].to_numpy(float)*scale
                position = alt_index + (control_index-0.5)*0.24
                axis.scatter(position+rng.normal(0, 0.035, len(values)), values, s=18,
                             color=CONTROL_COLORS[control], alpha=0.48, edgecolor="white", linewidth=0.3)
                add_median_iqr(axis, position, values, CONTROL_COLORS[control])
        axis.set_xticks(range(3), labels)
        axis.set_ylabel(ylabel, labelpad=3)
        final_title(axis, letter, title)
        final_style(axis, zero=True, grid_axis="y")
    handles = [Line2D([0], [0], marker="o", linestyle="", color=CONTROL_COLORS[c], label=CONTROL_LABELS[c]) for c in ["personal_center", "personal_range"]]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, facecolor="none", edgecolor="none", bbox_to_anchor=(0.5, 0.985))
    caption = ("Supplementary Figure S14 | Effects of occupant placement while command permission is held fixed. a,b, Changes in population and least-served mismatch. "
               "c, Personal-range coverage gain. d, Whole-building purchased-energy change. Points are matched placement comparisons; black marks show medians and interquartile ranges.")
    (RESULTS / "captions/Figure_S14.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S14", size_mm=(180, 145))

draw_s14()


In [ ]:
# Figure_S15: command-permission effects under fixed occupant placement.
def draw_s15():
    d = pd.read_csv(RESULTS / "global_extension/spatial_final/GLOBAL_PERMISSION_COMPARISONS.csv")
    d = d[d.status.eq("matched") & d.control.isin(["personal_center", "personal_range"])].copy()
    metrics = [("delta_mean_reference_mismatch_c", -1, "Population mismatch reduction (°C)", "Population service"),
               ("delta_fixed_tail_reference_mismatch_c", -1, "Least-served mismatch reduction (°C)", "Least-served group"),
               ("delta_modeled_interval_person_hour_coverage", 100, "Personal-range coverage gain (points)", "Range coverage"),
               ("delta_facility_purchased_pct", 1, "Whole-building energy change (%)", "Energy consequence")]
    transitions = [("building", "floor", "Building→\nfloor"),
                   ("floor", "native_zone", "Floor→\nzone"),
                   ("building", "native_zone", "Building→\nzone")]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 145/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.09, top=0.82, hspace=0.62, wspace=0.48)
    rng = np.random.default_rng(2026102715)
    for axis, (column, scale, ylabel, title), letter in zip(axes.flat, metrics, "abcd"):
        for transition_index, (reference, alternative, _) in enumerate(transitions):
            for control_index, control in enumerate(["personal_center", "personal_range"]):
                q = d[d.reference.eq(reference) & d.alternative.eq(alternative) & d.control.eq(control)]
                values = q[column].to_numpy(float)*scale
                position = transition_index + (control_index-0.5)*0.24
                axis.scatter(position+rng.normal(0, 0.035, len(values)), values, s=18,
                             color=CONTROL_COLORS[control], alpha=0.48, edgecolor="white", linewidth=0.3)
                add_median_iqr(axis, position, values, CONTROL_COLORS[control])
        axis.set_xticks(range(3), [t[2] for t in transitions])
        axis.set_ylabel(ylabel, labelpad=3)
        final_title(axis, letter, title)
        final_style(axis, zero=True, grid_axis="y")
    handles = [Line2D([0], [0], marker="o", linestyle="", color=CONTROL_COLORS[c], label=CONTROL_LABELS[c]) for c in ["personal_center", "personal_range"]]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, facecolor="none", edgecolor="none", bbox_to_anchor=(0.5, 0.985))
    caption = ("Supplementary Figure S15 | Effects of expanding command permission while occupant placement is held fixed. a,b, Changes in population and least-served mismatch. "
               "c, Personal-range coverage gain. d, Whole-building purchased-energy change. Points are matched permission comparisons; black marks show medians and interquartile ranges.")
    (RESULTS / "captions/Figure_S15.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S15", size_mm=(180, 145))

draw_s15()


In [ ]:
# Figure_S16: 72-event demand-response protocol.
def draw_s16():
    protocol = pd.read_csv(RESULTS / "global_extension/multi_event_dr_final/EVENT_PROTOCOL.csv")
    assert len(protocol) == 72 and protocol.climate.nunique() == 9
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.11, top=0.90, hspace=0.58, wspace=0.46)

    climates = CONTEXT_ORDER
    counts = protocol.groupby("climate").size().reindex(climates)
    axes[0, 0].bar(range(9), counts, color=["#4C82B8" if i % 2 == 0 else "#56A17A" for i in range(9)])
    axes[0, 0].set_xticks(range(9), [CONTEXT_LABELS[x] for x in climates], rotation=45, ha="right")
    axes[0, 0].set_ylabel("Event definitions")
    final_title(axes[0, 0], "a", "Eight events per climate")
    final_style(axes[0, 0], grid_axis="y")

    seasons = ["winter", "spring", "summer", "autumn"]
    table = protocol.groupby(["local_season", "load_level"]).size().unstack(fill_value=0).reindex(seasons)
    x = np.arange(4); width = 0.34
    axes[0, 1].bar(x-width/2, table["typical"], width, color="#76A7C5", label="Typical-load day")
    axes[0, 1].bar(x+width/2, table["high"], width, color="#D47C4A", label="High-load day")
    axes[0, 1].set_xticks(x, [s.title() for s in seasons])
    axes[0, 1].set_ylim(0, 14.5); axes[0, 1].set_yticks([0, 5, 10])
    axes[0, 1].set_ylabel("Event definitions")
    axes[0, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[0, 1], "b", "Two load days in every season")
    final_style(axes[0, 1], grid_axis="y")

    for season, color in zip(seasons, ["#76A7C5", "#63A788", "#D47C4A", "#C7A15B"]):
        values = protocol.loc[protocol.local_season.eq(season), "reference_mean_kw"]
        axes[1, 0].hist(values, bins=12, histtype="step", linewidth=1.8, color=color, label=season.title())
    axes[1, 0].set_xlabel("Reference whole-building power (kW)")
    axes[1, 0].set_ylim(0, 8.5)
    axes[1, 0].set_ylabel("Event definitions")
    axes[1, 0].legend(frameon=False, facecolor="none", edgecolor="none", ncol=2, loc="upper right")
    final_title(axes[1, 0], "c", "Reference load spans seasons")
    final_style(axes[1, 0], grid_axis="y")

    ax = axes[1, 1]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    stages = [(0.80, PALE_BLUE, "Baseline"),
              (0.60, "#DCEAF3", "Preconditioning · 2 h"),
              (0.40, PALE_ORANGE, "Grid event · 2 h"),
              (0.20, PALE_GREEN, "Recovery · 48 h")]
    for centre, color, text_label in stages:
        ax.add_patch(FancyBboxPatch((0.15, centre-0.065), 0.70, 0.13,
                                    boxstyle="round,pad=0.01,rounding_size=0.02",
                                    facecolor=color, edgecolor="#777777", linewidth=0.8))
        ax.text(0.50, centre, text_label, ha="center", va="center")
    for upper, lower in zip([0.80, 0.60, 0.40], [0.60, 0.40, 0.20]):
        ax.annotate("", xy=(0.50, lower+0.075), xytext=(0.50, upper-0.075),
                    arrowprops=dict(arrowstyle="->", color="#777777", lw=0.8))
    final_title(ax, "d", "Event and recovery accounting")

    caption = ("Supplementary Figure S16 | Demand-response event protocol. a, Eight event definitions in each of nine climates. b, Typical- and high-load days in each local season. "
               "c, Reference whole-building power across the selected event windows. d, Two-hour preconditioning, two-hour grid event and 48-hour recovery review used for each action.")
    (RESULTS / "captions/Figure_S16.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S16", size_mm=(180, 136))

draw_s16()


In [ ]:
# Figure_S17: strict factor-1.00 response across 72 events.
def draw_s17():
    root = RESULTS / "global_extension/multi_event_dr_final/summary"
    grouped = pd.read_csv(root / "EVENT_RESPONSE_GROUP_SUMMARY.csv")
    paired = pd.read_csv(root / "EVENT_RESPONSE_PAIRED_CONTROL_SUMMARY.csv")
    grouped = grouped[np.isclose(grouped.event_service_limit_factor, 1.0)]
    paired = paired[np.isclose(paired.main_service_factor, 1.0)]
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.11, top=0.90, hspace=0.60, wspace=0.47)

    c = grouped[grouped.group_dimension.eq("control")].set_index("group_value").reindex(CONTROL_ORDER)
    x = np.arange(3); width = 0.34
    axes[0, 0].bar(x-width/2, c.qualified_positive_event_fraction*100, width, color="#4C82B8", label="Event qualified")
    axes[0, 0].bar(x+width/2, c.qualified_positive_full_window_fraction*100, width, color=ACCENT_PURPLE, label="Full-window net-positive")
    axes[0, 0].set_xticks(x, ["Uniform\naverage", "Personal\ncentre", "Personal\nrange"])
    axes[0, 0].set_ylim(0, 112); axes[0, 0].set_ylabel("Positive responses (%)")
    axes[0, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[0, 0], "a", "Service limits narrow the usable response set")
    final_style(axes[0, 0], grid_axis="y")

    actions = ["uniform_0.5", "uniform_1.0", "protected_1.0", "precondition_0.5_uniform_1.0", "precondition_0.5_protected_1.0"]
    labels = ["Uniform\n+0.5 °C", "Uniform\n+1.0 °C", "Protected\n+1.0 °C", "Preconditioned\n+ uniform", "Preconditioned\n+ protected"]
    a = grouped[grouped.group_dimension.eq("action_id")].set_index("group_value").reindex(actions)
    y = np.arange(5)[::-1]
    axes[0, 1].barh(y, a.qualified_positive_event_fraction*100, color="#4C82B8")
    axes[0, 1].barh(y, a.qualified_positive_full_window_fraction*100, color=ACCENT_PURPLE, height=0.38)
    axes[0, 1].set_yticks(y, labels); axes[0, 1].tick_params(axis="y", length=0, pad=4)
    axes[0, 1].set_xlim(0, 78); axes[0, 1].set_xticks([0, 20, 40, 60])
    axes[0, 1].set_xlabel("Positive responses (%)")
    final_title(axes[0, 1], "b", "Response by action design")
    final_style(axes[0, 1], grid_axis="x")

    p = paired[paired.group_dimension.eq("overall")].set_index("alternative_control").reindex(["personal_center", "personal_range"])
    categories = ["alternative_only_event_qualified", "uniform_only_event_qualified", "both_event_qualified", "neither_event_qualified"]
    cat_labels = ["Demand-guided\nonly", "Uniform only", "Both", "Neither"]
    bottom = np.zeros(2)
    for column, label, color in zip(categories, cat_labels, ["#56A17A", "#D47C4A", "#76A7C5", "#D8D8D5"]):
        values = p[column].to_numpy()/p.pair_count.to_numpy()*100
        axes[1, 0].bar([0, 1], values, bottom=bottom, color=color, label=label)
        bottom += values
    axes[1, 0].set_xticks([0, 1], ["Personal\ncentre", "Personal\nrange"])
    axes[1, 0].set_ylim(0, 175); axes[1, 0].set_yticks([0, 50, 100]); axes[1, 0].set_ylabel("Matched event pairs (%)")
    axes[1, 0].legend(frameon=False, facecolor="none", edgecolor="none", ncol=2, loc="upper center", bbox_to_anchor=(0.5, 0.985), columnspacing=0.9)
    final_title(axes[1, 0], "c", "Demand information changes eligibility")
    final_style(axes[1, 0], grid_axis="y")

    seasons = ["winter", "spring", "summer", "autumn"]
    ss = grouped[grouped.group_dimension.eq("local_season")].set_index("group_value").reindex(seasons)
    axes[1, 1].bar(np.arange(4)-width/2, ss.qualified_positive_event_fraction*100, width, color="#4C82B8")
    axes[1, 1].bar(np.arange(4)+width/2, ss.qualified_positive_full_window_fraction*100, width, color=ACCENT_PURPLE)
    axes[1, 1].set_xticks(range(4), [s.title() for s in seasons])
    axes[1, 1].set_ylim(0, 90); axes[1, 1].set_ylabel("Positive responses (%)")
    final_title(axes[1, 1], "d", "Response varies by season")
    final_style(axes[1, 1], grid_axis="y")

    caption = ("Supplementary Figure S17 | Service-qualified response at the strict event service factor of 1.00. a, Positive event and full-window response by control. "
               "b, Response by action design. c, Paired event eligibility for each demand-guided control relative to Uniform average. d, Response across local seasons.")
    (RESULTS / "captions/Figure_S17.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S17", size_mm=(180, 136))

draw_s17()


In [ ]:
# Figure_S18: event tolerance, recovery accounting and terminal residuals.
def draw_s18():
    root = RESULTS / "global_extension/multi_event_dr_final"
    grouped = pd.read_csv(root / "summary/EVENT_RESPONSE_GROUP_SUMMARY.csv")
    scores = pd.read_csv(root / "EVENT_RESPONSE_SCORES.csv")
    terminal = pd.read_csv(root / "summary/EVENT_TERMINAL_RECOVERY_GROUP_SUMMARY.csv")
    overall = grouped[grouped.group_dimension.eq("overall")].sort_values("event_service_limit_factor")
    factors = overall.event_service_limit_factor.to_numpy()
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.11, top=0.90, hspace=0.60, wspace=0.47)

    axes[0, 0].plot(factors, overall.qualified_positive_event_fraction*100, "o-", color="#4C82B8", linewidth=1.8, label="Event qualified")
    axes[0, 0].plot(factors, overall.qualified_positive_full_window_fraction*100, "o-", color=ACCENT_PURPLE, linewidth=1.8, label="Full-window net-positive")
    axes[0, 0].axvline(1.0, color="#777777", linestyle="--", linewidth=0.8)
    axes[0, 0].set_xticks(factors, [f"{x:.2f}" for x in factors])
    axes[0, 0].set_xlabel("Event service-limit factor")
    axes[0, 0].set_ylabel("Positive responses (%)")
    axes[0, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper right")
    axes[0, 0].set_ylim(50, 90)
    final_title(axes[0, 0], "a", "Response versus event tolerance")
    final_style(axes[0, 0], grid_axis="y")

    q = scores[np.isclose(scores.event_service_limit_factor, 1.0) & scores.qualified_positive_full_window_response].copy()
    assert len(q) == 3579
    recovery_only = q.recovery_electricity_change_kwh / q.event_electricity_reduction_kwh * 100
    combined = (q.preconditioning_electricity_change_kwh + q.recovery_electricity_change_kwh) / q.event_electricity_reduction_kwh * 100
    assert np.isclose(np.median(recovery_only), 53.628970662537, rtol=0, atol=1e-9)
    assert np.isclose(np.median(combined), 65.563459834124, rtol=0, atol=1e-9)
    parts = axes[0, 1].violinplot([recovery_only, combined], positions=[0, 1], widths=0.72, showextrema=False, showmedians=True)
    for body, color in zip(parts["bodies"], ["#56A17A", ACCENT_PURPLE]):
        body.set_facecolor(color); body.set_alpha(0.72)
    parts["cmedians"].set_color("black")
    axes[0, 1].set_xticks([0, 1], ["Recovery\nonly", "Precondition\n+ recovery"])
    axes[0, 1].set_ylabel("Share of event electricity saving (%)")
    axes[0, 1].text(0, np.median(recovery_only)+4, "53.6%", ha="center")
    axes[0, 1].text(1, np.median(combined)+4, "65.6%", ha="center")
    final_title(axes[0, 1], "b", "Recovery shares of event saving")
    final_style(axes[0, 1], grid_axis="y")

    full = q.precondition_event_recovery_net_reduction_kwh
    event_power = q.event_mean_power_reduction_kw
    axes[1, 0].scatter(event_power, full, s=17, color="#8066A8", alpha=0.35, edgecolor="none")
    axes[1, 0].set_xlabel("Event power reduction (kW)")
    axes[1, 0].set_ylabel("Full-window net reduction (kWh)")
    final_title(axes[1, 0], "c", "Event and full-window response")
    final_style(axes[1, 0], zero=True, grid_axis="y")

    t = terminal[(terminal.audit_window.eq("final_recovery_hour")) & terminal.group_dimension.eq("overall")].iloc[0]
    labels = ["Operative\ntemperature", "Whole-building\nelectricity"]
    med = [t.operative_temperature_c_abs_median, t.facility_electricity_kw_abs_median]
    p95 = [t.operative_temperature_c_abs_p95, t.facility_electricity_kw_abs_p95]
    display_med = [med[0], med[1]/10]; display_p95 = [p95[0], p95[1]/10]
    x = np.arange(2); width = 0.34
    axes[1, 1].bar(x-width/2, display_med, width, color="#56A17A", label="Median")
    axes[1, 1].bar(x+width/2, display_p95, width, color="#8066A8", label="95th percentile")
    axes[1, 1].set_xticks(x, labels)
    axes[1, 1].set_ylabel("Residual (°C; electricity ÷10 kW)")
    axes[1, 1].set_ylim(0, 0.052)
    axes[1, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    final_title(axes[1, 1], "d", "Residuals at recovery hour 48")
    final_style(axes[1, 1], grid_axis="y")

    caption = ("Supplementary Figure S18 | Transient event tolerance and recovery accounting. a, Event-qualified and full-window net-positive response from the strict factor 1.00 through factors 1.05–1.20. "
               "b, Recovery-only electricity and the combined preconditioning-plus-recovery share of event electricity saving among full-window net-positive responses at factor 1.00; medians are 53.6% and 65.6%, respectively. c, Event power and full-window net electricity reduction. d, Median and 95th-percentile absolute operative-temperature and whole-building-electricity residuals at recovery hour 48; electricity is divided by ten only for the shared display scale.")
    (RESULTS / "captions/Figure_S18.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S18", size_mm=(180, 136))

draw_s18()


In [ ]:
# Figure_S19: implementation resources and declared sensitivity inputs.
def draw_s19():
    root = RESULTS / "global_extension/implementation_value_final"
    catalog = pd.read_csv(root / "IMPLEMENTATION_RESOURCE_CATALOG.csv")
    scenarios = pd.read_csv(root / "IMPLEMENTATION_RESOURCE_SCENARIOS.csv")
    commands = pd.read_csv(root / "COMMAND_SCOPE_REQUIREMENTS.csv")
    spatial = pd.read_csv(root / "GLOBAL_SPATIAL_IMPLEMENTATION_BOUNDARIES.csv")
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.18, right=0.985, bottom=0.11, top=0.90, hspace=0.60, wspace=0.50)

    plain = ["Demand feedback", "Software setup", "Maintenance", "Occupant\nreassignment", "Command scope", "Local hardware"]
    y = np.arange(6)[::-1]
    required = catalog.required_in_every_deployment.astype(bool).to_numpy()
    axes[0, 0].scatter(required.astype(int), y, s=65,
                       color=np.where(required, "#4C82B8", "#8066A8"), edgecolor="white", linewidth=0.5)
    axes[0, 0].set_yticks(y, plain)
    axes[0, 0].set_xticks([0, 1], ["Optional or\ncontext dependent", "Core resource"])
    axes[0, 0].set_xlim(-0.35, 1.35)
    final_title(axes[0, 0], "a", "Six implementation resources")
    final_style(axes[0, 0], grid_axis="x")

    groups = commands.set_index("command_scope").reindex(["building", "floor", "native_zone"])
    axes[0, 1].bar(range(3), groups.independent_command_groups, color=["#73777A", "#C7A15B", "#56A17A"])
    axes[0, 1].set_xticks(range(3), ["Building", "Floor", "Zone"])
    axes[0, 1].set_ylabel("Independent command groups")
    final_title(axes[0, 1], "b", "Command groups by scope")
    final_style(axes[0, 1], grid_axis="y")

    named = scenarios[scenarios.kind.ne("sensitivity_grid")].copy()
    if named.empty:
        named = scenarios.copy()
    for index, wage in enumerate(["median", "mean"]):
        values = scenarios.loc[scenarios.wage_profile.eq(wage), "annual_resource_cost_usd"]
        axes[1, 0].hist(values, bins=16, histtype="step", linewidth=1.8,
                        color=["#4C82B8", "#D47C4A"][index], label=wage.title()+" wage")
    axes[1, 0].set_xlabel("Annualized implementation input (US$ yr⁻¹)")
    axes[1, 0].set_ylim(0, 24)
    axes[1, 0].set_ylabel("Declared scenarios")
    axes[1, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper center", ncol=2, bbox_to_anchor=(0.5, 0.985))
    final_title(axes[1, 0], "c", "Declared resource scenarios")
    final_style(axes[1, 0], grid_axis="y")

    q = spatial[spatial.resource_category.eq("command_scope")].drop_duplicates(["climate", "composition", "control", "layout", "reference", "alternative"])
    axes[1, 1].scatter(q.incremental_resource_quantity, -q.delta_mean_reference_mismatch_c,
                       s=18, color="#56A17A", alpha=0.42, edgecolor="none")
    axes[1, 1].set_xlim(0, 16); axes[1, 1].set_xticks([0, 5, 10, 15])
    axes[1, 1].set_xlabel("Additional command groups")
    axes[1, 1].set_ylabel("Population mismatch reduction (°C)")
    final_title(axes[1, 1], "d", "Benefit and command effort")
    final_style(axes[1, 1], zero=True, grid_axis="y")

    caption = ("Supplementary Figure S19 | Implementation resources and declared sensitivity inputs. a, Six resource categories—demand feedback and profile estimation, software setup, maintenance, occupant reassignment, command scope and local hardware—and whether each is required in every deployment. "
               "b, Independent command groups under building-, floor- and zone-level permission. c, Distribution of annualized resource inputs across the declared median- and mean-wage scenarios. d, Additional command groups and population-mismatch reduction in the spatial comparisons.")
    (RESULTS / "captions/Figure_S19.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S19", size_mm=(180, 136))

draw_s19()


In [ ]:
# Figure_S20: non-monetized value boundaries and decision thresholds.
def draw_s20():
    root = RESULTS / "global_extension/implementation_value_final"
    boundaries = pd.read_csv(root / "GLOBAL_SERVICE_RESOURCE_VALUE_BOUNDARIES.csv")
    audit = json.loads((root / "IMPLEMENTATION_VALUE_AUDIT.json").read_text(encoding="utf-8"))
    assert not audit["local_energy_prices_assigned"] and not audit["annual_dr_revenue_assigned"]
    direct = boundaries.drop_duplicates(["climate", "composition", "alternative_control", "target_service_gain_c", "delta_facility_purchased_kwh_year"])
    scenarios = boundaries.drop_duplicates("scenario_id")
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.11, top=0.90, hspace=0.60, wspace=0.48)

    for control in ["personal_center", "personal_range"]:
        q = direct[direct.alternative_control.eq(control)]
        axes[0, 0].scatter(q.target_service_gain_c, q.delta_facility_purchased_kwh_year/1000,
                           s=22, color=CONTROL_COLORS[control], alpha=0.65,
                           edgecolor="white", linewidth=0.35, label=CONTROL_LABELS[control])
    axes[0, 0].set_xlabel("Population mismatch reduction (°C)")
    axes[0, 0].set_ylabel("Whole-building energy change (MWh)")
    axes[0, 0].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper left")
    axes[0, 0].set_ylim(-10, 60)
    final_title(axes[0, 0], "a", "Physical service–energy boundary")
    final_style(axes[0, 0], zero=True, grid_axis="y")

    for index, wage in enumerate(["median", "mean"]):
        values = scenarios.loc[scenarios.wage_profile.eq(wage), "illustrative_annual_resource_cost_usd"]
        axes[0, 1].hist(values, bins=15, histtype="step", linewidth=1.8,
                        color=["#4C82B8", "#D47C4A"][index], label=wage.title()+" wage")
    axes[0, 1].set_xticks([0, 1000, 2000])
    axes[0, 1].set_xlabel("Annualized input (US$ yr⁻¹)")
    axes[0, 1].set_ylim(0, 23)
    axes[0, 1].set_ylabel("Declared scenarios")
    axes[0, 1].legend(frameon=False, facecolor="none", edgecolor="none", loc="upper right")
    final_title(axes[0, 1], "b", "Implementation-input range")
    final_style(axes[0, 1], grid_axis="y")

    axes[1, 0].scatter(direct.delta_facility_electric_kwh_year/1000,
                       direct.delta_facility_gas_hhv_kwh_year/1000,
                       s=20, color="#8066A8", alpha=0.50, edgecolor="white", linewidth=0.3)
    axes[1, 0].set_xlabel("Electricity coefficient (MWh yr⁻¹)")
    axes[1, 0].set_ylabel("Gas coefficient (MWh yr⁻¹ HHV)")
    final_title(axes[1, 0], "c", "Energy-price coefficients")
    final_style(axes[1, 0], zero=True, grid_axis="y")

    ax = axes[1, 1]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    boxes = [(0.05, 0.62, PALE_GREEN, "Thermal service\nvalue"),
             (0.55, 0.62, PALE_BLUE, "Grid-response\nvalue"),
             (0.05, 0.18, PALE_ORANGE, "Implementation\nresources"),
             (0.55, 0.18, PALE_PURPLE, "Local energy\ncharge change")]
    for left, bottom, color, label in boxes:
        ax.add_patch(FancyBboxPatch((left, bottom), 0.38, 0.22,
                                    boxstyle="round,pad=0.015,rounding_size=0.025",
                                    facecolor=color, edgecolor="#777777", linewidth=0.8))
        ax.text(left+0.19, bottom+0.11, label, ha="center", va="center")
    ax.text(0.49, 0.73, "+", ha="center", va="center")
    ax.text(0.49, 0.29, "+", ha="center", va="center")
    ax.text(0.49, 0.51, "≥", ha="center", va="center")
    final_title(ax, "d", "Value must cover resources")

    caption = ("Supplementary Figure S20 | Decision boundaries before assigning local energy prices or annual grid-response revenue. a, Physical relation between population-mismatch reduction and whole-building energy change. "
               "b, Annualized implementation inputs across declared sensitivity scenarios. c, Electricity and gas coefficients that local prices could weight. d, Decision inequality linking thermal-service and grid-response value to implementation resources and local energy-charge change. No local prices, thermal-service monetary values or grid-response revenue are assigned.")
    (RESULTS / "captions/Figure_S20.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S20", size_mm=(180, 136))

draw_s20()


In [ ]:
# Figure_S21: solver-stability and reproducibility evidence.
def draw_s21():
    annual_primary = pd.read_csv(ROOT / "output/global_climate_matrix_fixed_v2_40iter/annual_solver_stability_100/ANNUAL_SOLVER_STABILITY.csv")
    annual_extension = pd.read_csv(ROOT / "output/global_climate_matrix_fixed_v2_40iter/annual_candidate_extension_solver_100/ANNUAL_SOLVER_STABILITY.csv")
    seasonal = pd.read_csv(ROOT / "output/global_seasonal_command_v1_stability_100/SEASONAL_SOLVER_STABILITY.csv")
    spatial = pd.read_csv(ROOT / "output/global_spatial_extension_v1/solver_100/ANNUAL_SOLVER_STABILITY.csv")
    annual = pd.concat([annual_primary, annual_extension], ignore_index=True)
    assert [len(annual_primary), len(annual_extension), len(seasonal), len(spatial)] == [62, 70, 63, 133]
    maxima_t = [annual.operative_temperature_max_abs_difference_c.max(),
                seasonal.operative_temperature_max_abs_difference_c.max(),
                spatial.operative_temperature_max_abs_difference_c.max()]
    assert np.isclose(maxima_t[0], 0.000225343, rtol=0, atol=5e-10)
    assert np.isclose(maxima_t[1], 0.000671796, rtol=0, atol=5e-10)
    assert np.isclose(maxima_t[2], 0.001511342, rtol=0, atol=5e-10)
    annual_energy = annual.total_purchased_energy_relative_difference.abs().max()
    spatial_energy = spatial.facility_purchased_energy_relative_difference.abs().max()
    assert np.isclose(annual_energy, 1.72735e-7, rtol=0, atol=5e-12)
    assert np.isclose(spatial_energy, 3.33489e-6, rtol=0, atol=5e-11)
    fig, axes = plt.subplots(2, 2, figsize=(180/25.4, 136/25.4))
    fig.subplots_adjust(left=0.15, right=0.985, bottom=0.11, top=0.90, hspace=0.62, wspace=0.48)

    labels = ["Annual\nprimary", "Annual\nextension", "Seasonal", "Spatial"]
    counts = [62, 70, 63, 133]
    axes[0, 0].bar(range(4), counts, color=["#4C82B8", "#76A7C5", "#C7A15B", "#56A17A"])
    axes[0, 0].set_xticks(range(4), labels)
    axes[0, 0].set_ylabel("Rerun comparisons")
    for index, value in enumerate(counts): axes[0, 0].text(index, value+3, str(value), ha="center")
    final_title(axes[0, 0], "a", "Targeted solver reruns")
    final_style(axes[0, 0], grid_axis="y")

    axes[0, 1].bar(range(3), np.asarray(maxima_t)*1000, color=["#4C82B8", "#C7A15B", "#56A17A"])
    axes[0, 1].set_xticks(range(3), ["Annual", "Seasonal", "Spatial"])
    axes[0, 1].set_ylabel("Maximum temperature gap (m°C)")
    final_title(axes[0, 1], "b", "Temperature stability")
    final_style(axes[0, 1], grid_axis="y")

    energy_values = [annual_energy*100, spatial_energy*100]
    axes[1, 0].bar([0, 1], energy_values, color=["#4C82B8", "#56A17A"])
    axes[1, 0].set_xticks([0, 1], ["Annual", "Spatial"])
    axes[1, 0].set_ylabel("Maximum energy gap (%)")
    for index, value in enumerate(energy_values): axes[1, 0].text(index, value*1.04, f"{value:.7f}%", ha="center")
    final_title(axes[1, 0], "c", "Energy results remain unchanged")
    final_style(axes[1, 0], grid_axis="y")

    pass_rates = [annual.trajectory_stable_within_declared_tolerance.mean()*100,
                  seasonal.trajectory_stable_within_declared_tolerance.mean()*100,
                  spatial.trajectory_stable_within_declared_tolerance.mean()*100]
    axes[1, 1].bar(range(3), pass_rates, color=["#4C82B8", "#C7A15B", "#56A17A"])
    axes[1, 1].set_xticks(range(3), ["Annual", "Seasonal", "Spatial"])
    axes[1, 1].set_ylim(0, 108); axes[1, 1].set_ylabel("Within tolerance (%)")
    for index, value in enumerate(pass_rates): axes[1, 1].text(index, value+1.5, f"{value:.0f}%", ha="center")
    final_title(axes[1, 1], "d", "All reruns meet tolerance")
    final_style(axes[1, 1], grid_axis="y")
    for axis in axes.flat:
        axis.title.set_x(0.02)

    caption = ("Supplementary Figure S21 | Solver-stability evidence from targeted reruns at 100 HVAC iterations. a, Annual primary (62), annual candidate-extension (70), seasonal (63) and spatial (133) comparisons. "
               "b, Maximum operative-temperature trajectory differences: 0.000225343 °C annually, 0.000671796 °C seasonally and 0.001511342 °C spatially. c, Maximum relative whole-building purchased-energy differences of 0.0000173% annually and 0.000333% spatially. d, Share of targeted trajectories within the declared temperature and energy tolerances.")
    (RESULTS / "captions/Figure_S21.txt").write_text(caption, encoding="utf-8")
    save_plot(fig, "Figure_S21", size_mm=(180, 136))

draw_s21()
